# Mini-hackathon: is super-resolution a fit for *your* decision?

*SSTA workshop, 27 November 2026.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/hackathon/scenario/hackathon_scenario.ipynb)

**Your team writes the brief.** Pick a decision that someone in your agencies really makes and
that an image could inform. Then walk through what it takes to judge whether super-resolved
(SR) Sentinel-2 fits: the area, the images that exist, the money, what the pictures show, what
we measured for similar tasks, and how you would test it at home.

By 14:15, answer on one slide: **pilot it**, **test it first** or **not a fit**, with your test
plan. There is no leaderboard. "Super-resolution is not a fit here" is a good answer.

| Part | What you do | When |
|---|---|---|
| 0 | Start the notebook with one click | 11:15 |
| 1 | Write your scenario | 11:15–11:50 |
| 2 | Choose your area and dates. An area of your own: send it **before lunch** | 11:50–12:00 |
| 3 | What would the images cost? | 13:00 |
| 4 | Look: which image would you use? | 13:25 |
| 5 | Cheat sheet and next steps: how would you test it? | 13:55 |
| 6 | Your slide | 14:15 |

**No coding needed.** You type, click and discuss.

## 0. Start with one click

1. In the top menu, choose **Runtime → Run all**.
2. Colab warns that Google did not write this notebook. Choose **Run anyway**.
3. Wait about a minute, then scroll down.

After that, use only the **controls**: text fields, menus, sliders and buttons. They act at
once and save your answers. If a control does nothing, click the ▶ to the left of its cell.
Keep this browser tab open over lunch.

**Working online?** One person shares their screen and operates the controls. Give out four
roles:

| Role | Job |
|---|---|
| Decision-maker | speaks for the office that decides |
| Field lead | keeps the numbers realistic: places, visits, money |
| Sceptic | asks "what if the picture is wrong?" |
| Rapporteur | fills the slide and presents |

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
# Helpers for the SSTA mini-hackathon notebooks.
# This file is pasted into a hidden cell of each notebook by build.py, so every notebook
# is self-contained. Edit it here, then re-run build.py.

import json
import math
import os
import shutil
import sys
import time
import urllib.request
import zipfile
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from PIL import Image

# --------------------------------------------------------------------------- look

COLOURS = {"s2": "#9e9e9e", "stage1": "#8fd3b6", "stage2": "#1f8a70", "hr": "#007db7",
           "random": "#d9d9d9", "best": "#4d4d4d", "perfect": "#4d4d4d"}
SHORT = {"s2": "Sentinel-2", "stage1": "SR stage 1", "stage2": "SR stage 2",
         "hr": "Real high-res", "s2_10m": "Sentinel-2 as delivered (10 m)"}
HIT_C, MISS_C = "#1a9850", "#d73027"
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# --------------------------------------------------------------------------- data

DATA_ROOT = Path("/content/hackathon_data") if Path("/content").exists() else Path("hackathon_data")


def _fetch(url, dest, token=None):
    """Download url to dest with a simple progress line. url may also be a local path."""
    if not str(url).startswith(("http://", "https://")):
        shutil.copy(url, dest)
        return
    req = urllib.request.Request(url, headers={"User-Agent": "ssta-hackathon"})
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        done, t0, last = 0, time.time(), 0.0
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if time.time() - last > 2:
                last = time.time()
                pct = f"{done / total:.0%}" if total else ""
                print(f"\r  {done / 1e6:,.0f} MB {pct}   ", end="")
        print(f"\r  {done / 1e6:,.0f} MB in {time.time() - t0:.0f} s   ")


def get_task(task, base_url, extras=(), token=None):
    """Download and unzip <base_url>/<task>.zip (and optional extra zips) once; return a Task."""
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name, required in [(task, True)] + [(e, False) for e in extras]:
        marker = DATA_ROOT / f".{name}.done"
        if marker.exists():
            continue
        url = f"{str(base_url).rstrip('/')}/{name}.zip"
        tmp = DATA_ROOT / f"{name}.zip"
        print(f"Downloading {name}.zip ...")
        try:
            _fetch(url, tmp, token)
            with zipfile.ZipFile(tmp) as z:
                z.extractall(DATA_ROOT)
            tmp.unlink()
            marker.touch()
        except Exception as e:  # extras are optional (e.g. restricted high-res images)
            if required:
                raise RuntimeError(f"Could not get {url}: {e}. Ask a facilitator.") from e
            print(f"  (skipped {name}: {e}; the notebook works without it)")
    return Task(DATA_ROOT / task)


class Task:
    def __init__(self, root):
        self.root = Path(root)
        self.meta = json.loads((self.root / "meta.json").read_text())
        self.inputs = [i["key"] for i in self.meta["inputs"]]
        self.labels = {i["key"]: i["label"] for i in self.meta["inputs"]}
        self.scenes = pd.DataFrame(self.meta["scenes"]).set_index("id")
        self.px = float(self.meta.get("pixel_m", 2.5))
        first = self.scenes.index[0]
        self.has_image = {k: (self.root / "img" / first / f"{k}.jpg").exists() for k in self.inputs}

    def __repr__(self):
        return f"<{self.meta['title']}: {len(self.scenes)} scenes, inputs {self.inputs}>"

    @lru_cache(maxsize=48)
    def _npz(self, sid):
        return np.load(self.root / "arr" / f"{sid}.npz")

    @lru_cache(maxsize=32)
    def arr(self, sid, key):
        """One array of a scene (decompressed once and kept; do not change it in place)."""
        return self._npz(sid)[key]

    def keys(self, sid):
        return self._npz(sid).files

    def img(self, sid, key, pre=False):
        """Display RGB (uint8) on the 2.5 m grid. key = an input or 's2_10m' (shown as 10 m blocks)."""
        suffix = "_pre" if pre else ""
        if key == "s2_10m":
            im = Image.open(self.root / "img" / sid / f"s2_10m{suffix}.png").convert("RGB")
            h, w = int(self.scenes.loc[sid, "h"]), int(self.scenes.loc[sid, "w"])
            return np.asarray(im.resize((w, h), Image.NEAREST))
        p = self.root / "img" / sid / f"{key}{suffix}.jpg"
        if not p.exists():
            return None
        return np.asarray(Image.open(p).convert("RGB"))

    @lru_cache(maxsize=24)
    def _view_u8(self, sid, key, stretch, pre):
        im = _view(self.img(sid, key, pre=pre), stretch)
        return None if im is None else (im * 255).round().astype(np.uint8)

    def view(self, sid, key, stretch="auto", pre=False):
        """The image as shown (contrast-stretched once per image and kept), floats 0-1."""
        im = self._view_u8(sid, key, stretch, pre)
        return None if im is None else im / 255.0


def autostretch(rgb, lo=2, hi=98):
    """Per-image contrast stretch for viewing only (the models saw the original values)."""
    if rgb is None:
        return None
    x = rgb.astype(np.float32)
    out = np.empty_like(x)
    for c in range(3):
        a, b = np.percentile(x[..., c], [lo, hi])
        out[..., c] = np.clip((x[..., c] - a) / max(b - a, 1), 0, 1)
    return out


def _view(rgb, stretch):
    return autostretch(rgb) if stretch == "auto" else (None if rgb is None else rgb / 255.0)


def _outline(mask):
    m = mask.astype(bool)
    edge = np.zeros_like(m)
    edge[1:, :] |= m[1:, :] != m[:-1, :]
    edge[:, 1:] |= m[:, 1:] != m[:, :-1]
    return edge & m


def _overlay(ax, mask, colour, alpha=1.0):
    rgba = np.zeros(mask.shape + (4,))
    rgba[mask] = list(plt.matplotlib.colors.to_rgb(colour)) + [alpha]
    ax.imshow(rgba, interpolation="nearest")


def _blank(ax, text="not available"):
    ax.text(0.5, 0.5, text, ha="center", va="center", color="#777", transform=ax.transAxes)
    ax.set_xticks([]), ax.set_yticks([])


def area_around(lat, lon, km=2.0):
    """A km x km box [W, S, E, N] centred on (lat, lon); the same as sr_client.area_around."""
    dlat = km / 2 / 111.32
    dlon = km / 2 / (111.32 * max(math.cos(math.radians(lat)), 0.01))
    return [round(v, 6) for v in (lon - dlon, lat - dlat, lon + dlon, lat + dlat)]


# --------------------------------------------------------------------------- decisions

def expected_hits(score, hit, k):
    """Hits among the top-k by score. Ties at the cut-off are shared out at their hit rate,
    so a model that gives many places the same score is not lucky or unlucky by chance."""
    score = np.asarray(score, float)
    hit = np.asarray(hit, bool)
    k = min(int(k), len(score))
    if k == 0:
        return 0.0
    v = np.sort(score)[::-1][k - 1]
    above, tied = score > v, score == v
    return float(hit[above].sum() + (k - above.sum()) * hit[tied].mean())


def hits_curve(score, hit):
    """Expected hits among the top-k places for k = 1 .. n (ties share their hit rate).
    hits_curve(score, hit)[k - 1] equals expected_hits(score, hit, k)."""
    score = np.asarray(score, float)
    hit = np.asarray(hit, float)
    order = np.argsort(-score, kind="stable")
    _, group, size = np.unique(-score[order], return_inverse=True, return_counts=True)
    rate = np.bincount(group, weights=hit[order]) / size
    return np.cumsum(rate[group])


def visits_needed(score, hit, target):
    """The smallest budget whose top places hold at least `target` hits (None: never)."""
    curve = hits_curve(score, hit)
    k = int(np.searchsorted(curve, target - 1e-9))
    return k + 1 if k < len(curve) else None


def screen_then_confirm(screen, confirm, hit, k, m, seed=0):
    """Two steps: shortlist the top-m places with a cheap input (`screen`), look only at
    those with a better input (`confirm`) and visit its top-k. Returns the expected hits."""
    hit = np.asarray(hit, bool)
    short = top_k(screen, max(int(m), int(k)), seed)
    return expected_hits(np.asarray(confirm, float)[short], hit[short], k)


def where_table(units, preds, hit, k, by="group", labels=None):
    """Where the visits would go: places, true hits and the visits of each input (picked on
    the mean score of its models), per area. Shows whether an area is left out."""
    hit = np.asarray(hit, bool)
    out = pd.DataFrame({"places": units.groupby(by).size(),
                        "true hits": units[hit].groupby(by).size()}).fillna(0).astype(int)
    for inp, p in preds.items():
        picks = units.iloc[top_k(np.atleast_2d(p).mean(0), k)]
        out[f"visits: {(labels or {}).get(inp, inp)}"] = picks.groupby(by).size()
    out = out.fillna(0).astype(int).sort_values("true hits", ascending=False)
    out.index.name = None
    return out


def top_k(score, k, seed=0):
    """Indices of the top-k (ties broken at random, reproducibly)."""
    rng = np.random.default_rng(seed)
    jitter = rng.random(len(score)) * 1e-9
    return np.argsort(-(np.asarray(score, float) + jitter))[:k]


def decide(units, preds, hit, budget, cost_wasted=1.0, cost_missed=1.0, labels=None):
    """Score the decision 'visit the top <budget> places' for every input.

    units: DataFrame, one row per place.  preds: {input: array (n_models, n_places)}.
    hit: bool array, which places are worth visiting (from the reference labels).
    Returns a DataFrame with one row per input plus 'random' and 'perfect'."""
    hit = np.asarray(hit, bool)
    n_hit, n = int(hit.sum()), len(hit)
    k = min(int(budget), n)
    rows = []

    def row(name, hits_per_model):
        h = np.asarray(hits_per_model, float)
        m = h.mean()
        rows.append({
            "input": name,
            "hits": round(m, 1),
            "range over models": f"{h.min():.0f}–{h.max():.0f}" if len(h) > 1 else "",
            "hit rate": f"{m / k:.0%}" if k else "",
            "wasted visits": round(k - m, 1),
            "missed places": round(n_hit - m, 1),
            "cost": round((k - m) * cost_wasted + (n_hit - m) * cost_missed, 1),
        })

    for inp, p in preds.items():
        row(inp, [expected_hits(pm, hit, k) for pm in np.atleast_2d(p)])
    row("random", [k * n_hit / max(n, 1)])
    row("perfect", [min(k, n_hit)])
    out = pd.DataFrame(rows).set_index("input")
    if labels:
        out.index = [labels.get(i, i) for i in out.index]
    print(f"{n:,} places, {n_hit:,} of them are hits ({n_hit / max(n, 1):.0%}); budget {k}.")
    return out


def plot_decision(res, budget, title="", guess=None):
    """Bars of hits per input. guess: {row name: your bet}, drawn as a black line."""
    names = list(res.index)
    keys = [next((k for k, v in SHORT.items() if v == nm or k == nm), nm) for nm in names]
    cols = [COLOURS.get(k, "#999") for k in keys]
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(names) + 1))
    y = np.arange(len(names))[::-1]
    ax.barh(y, res["hits"], color=cols)
    for yi, nm, v in zip(y, names, res["hits"]):
        bet = (guess or {}).get(nm)      # keep the number clear of the bet line
        ax.text(max(v, bet if bet is not None else 0) + budget * 0.012, yi, f"{v:g}", va="center")
    shown = [(yi, guess[nm]) for yi, nm in zip(y, names) if guess and guess.get(nm) is not None]
    if shown:
        ax.vlines([g for _, g in shown], [yi - 0.46 for yi, _ in shown],
                  [yi + 0.46 for yi, _ in shown], color="black", lw=2.5, label="your bet")
        ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 1), fontsize=9)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_yticks(y, names)
    ax.set_xlim(0, budget * 1.12)
    ax.axvline(budget, color="#bbb", lw=1, ls="--")
    ax.set_xlabel(f"hits among the {budget} places you would visit")
    ax.set_title(title, loc="left")
    plt.tight_layout()
    return fig


def plot_strategies(rows, budget, title=""):
    """rows: dicts with name, hits, cost (USD a year) and colour. One bar per strategy,
    labelled with its hits and its imagery bill."""
    fig, ax = plt.subplots(figsize=(8.6, 0.55 * len(rows) + 1.1))
    y = np.arange(len(rows))[::-1]
    ax.barh(y, [r["hits"] for r in rows], color=[r["colour"] for r in rows])
    for yi, r in zip(y, rows):
        cost = "free" if r["cost"] < 0.5 else f"USD {r['cost']:,.0f} a year"
        ax.text(r["hits"] + budget * 0.01, yi, f"{r['hits']:.1f} hits · {cost}", va="center")
    ax.set_yticks(y, [r["name"] for r in rows])
    ax.set_xlim(0, budget * 1.75)
    ax.set_xticks(np.linspace(0, budget, 5).round().astype(int))
    ax.set_xlabel(f"hits among your {budget} visits (and the imagery bill for your whole area)")
    ax.set_title(title, loc="left")
    plt.tight_layout()
    return fig


def sweep(preds, hit, budgets=(5, 10, 20, 30, 50, 75, 100), labels=None, title=""):
    """Hit rate for a range of budgets: does the advantage of one input hold for small and large budgets?"""
    hit = np.asarray(hit, bool)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    budgets = [b for b in budgets if b <= len(hit)]
    for inp, p in preds.items():
        r = [np.mean([expected_hits(pm, hit, b) for pm in np.atleast_2d(p)]) / b for b in budgets]
        ax.plot(budgets, r, "o-", color=COLOURS.get(inp, "#999"), label=(labels or {}).get(inp, inp), lw=2)
    ax.plot(budgets, [hit.mean()] * len(budgets), "--", color="#999", label="random")
    ax.plot(budgets, [min(b, hit.sum()) / b for b in budgets], ":", color=COLOURS["best"], label="perfect")
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("budget (places visited)")
    ax.set_ylabel("share of visits that are hits")
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


def overlap_table(preds, k, labels=None, seed=0):
    """Share of the top-k places that two inputs have in common (picked on the mean score of
    each input's models, like the pictures below it)."""
    picks = {i: set(top_k(np.atleast_2d(p).mean(0), k, seed)) for i, p in preds.items()}
    names = list(picks)
    m = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            m.loc[a, b] = len(picks[a] & picks[b]) / max(k, 1)
    if labels:
        m.index = [labels.get(i, i) for i in names]
        m.columns = m.index
    try:
        return m.style.format("{:.0%}").background_gradient(cmap="Greens", vmin=0, vmax=1)
    except Exception:  # no jinja2: plain table
        return m.map(lambda v: f"{v:.0%}") if hasattr(m, "map") else m.applymap(lambda v: f"{v:.0%}")


def slide_numbers(res, scenario, budget, task_title, fname="my_slide_chart.png"):
    """Print the numbers for the team slide and save the bar chart as a PNG."""
    print(f"TASK: {task_title}")
    for k, v in scenario.items():
        print(f"{k.replace('_', ' ').upper()}: {v}")
    print()
    print(res[["hits", "range over models", "hit rate", "missed places", "cost"]].to_string())
    fig = plot_decision(res, budget, f"{scenario.get('decision', '')}")
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    print(f"\nSaved {fname} (in Colab: Files panel on the left, right-click > Download).")


# --------------------------------------------------------------------------- informal settlements

def ifs_f1_table(task, thr=0.5):
    """Pixel F1 of the informal-settlement class per input, mean and spread over the task models."""
    acc = {}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid")
        v = val & (lab != 255)
        t = (lab == 1) & v
        for inp in task.inputs:
            p = task.arr(sid, f"prob_{inp}") >= thr * 255
            tp = (p & t).sum((1, 2))
            fp = (p & ~t & v).sum((1, 2))
            fn = (~p & t).sum((1, 2))
            a = acc.setdefault(inp, np.zeros((3, len(tp))))
            a += [tp, fp, fn]
    paper = {r["input"]: r.get("paper") for r in task.meta["metric_table"]}
    rows = []
    for inp, (tp, fp, fn) in acc.items():
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1) * 100
        rows.append({"input": task.labels[inp], "F1 % (mean)": round(f1.mean(), 1),
                     "lowest model": round(f1.min(), 1), "highest model": round(f1.max(), 1),
                     "paper": paper.get(inp)})
    return pd.DataFrame(rows).set_index("input")


def ifs_units(task, cell_m=100, thr=0.5, min_valid=0.8):
    """Cut every tile into square blocks of cell_m metres. For each block: the true informal share
    and, per input and task model, the share of the block the model flags as informal."""
    c = max(4, int(round(cell_m / task.px)))
    rows, preds = [], {inp: [] for inp in task.inputs}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid") & (task.arr(sid, "label") != 255)
        H, W = lab.shape
        ny, nx = H // c, W // c
        if ny == 0 or nx == 0:
            continue
        def blocks(a):
            return a[..., :ny * c, :nx * c].reshape(a.shape[:-2] + (ny, c, nx, c)).sum((-3, -1))
        nv = blocks(val.astype(np.float32))
        ninf = blocks(((lab == 1) & val).astype(np.float32))
        for inp in task.inputs:
            p = (task.arr(sid, f"prob_{inp}") >= thr * 255) & val
            preds[inp].append((blocks(p.astype(np.float32)) / np.maximum(nv, 1)).reshape(p.shape[0], -1))
        for iy in range(ny):
            for ix in range(nx):
                rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": iy * c,
                             "x0": ix * c, "size": c, "valid": nv[iy, ix] / c / c,
                             "truth": ninf[iy, ix] / max(nv[iy, ix], 1)})
    units = pd.DataFrame(rows)
    preds = {k: np.concatenate(v, axis=1) for k, v in preds.items()}
    keep = units["valid"].values >= min_valid
    units = units[keep].reset_index(drop=True)
    preds = {k: v[:, keep] for k, v in preds.items()}
    print(f"{len(units):,} blocks of {c * task.px:.0f} m x {c * task.px:.0f} m in {units.scene.nunique()} tiles")
    return units, preds


def ifs_show(task, sid, inputs=None, window=None, stretch="auto", model=None, thr=0.5,
             box=None, title=None):
    """Top row: the images. Bottom row: what the task model finds (orange) and the reference outline (black)."""
    inputs = inputs or ["s2_10m"] + task.inputs
    lab = task.arr(sid, "label")
    y0, x0, s = window or (0, 0, max(lab.shape))
    sl = (slice(y0, y0 + s), slice(x0, x0 + s))
    ref_edge = _outline(lab == 1)[sl]
    fig, axes = plt.subplots(2, len(inputs), figsize=(2.6 * len(inputs), 5.6), squeeze=False)
    for j, key in enumerate(inputs):
        ax, bx = axes[0, j], axes[1, j]
        im = task.view(sid, key, stretch)
        if im is None:
            _blank(ax, "image not\ndownloaded")
        else:
            ax.imshow(im[sl], interpolation="nearest")
        ax.set_title(SHORT.get(key, key))
        pkey = "s2" if key == "s2_10m" else key
        if key == "s2_10m":
            _blank(bx, "(the model sees the\nupsampled version)")
        else:
            prob = task.arr(sid, f"prob_{pkey}")
            p = (prob[model] if model is not None else prob.mean(0)) / 255.0
            bx.imshow(np.ones(p[sl].shape + (3,)))
            _overlay(bx, p[sl] >= thr, "#e6550d", 0.85)
            _overlay(bx, ref_edge, "black")
            if box:
                for b in (ax, bx):
                    b.add_patch(Rectangle((box[1] - x0 - .5, box[0] - y0 - .5), box[2], box[2],
                                          fill=False, ec="#d73027", lw=2))
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    axes[1, 0].set_ylabel("model output")
    fig.suptitle(title or f"{sid}  ({task.scenes.loc[sid, 'group']})", x=0.01, ha="left")
    fig.legend(handles=[Patch(color="#e6550d", label="flagged as informal (mean of models)"),
                        Patch(fc="white", ec="black", label="reference outline")],
               loc="lower center", ncol=2, frameon=False)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


def ifs_gallery(task, units, idx, key="stage2", stretch="auto", ctx=128, title="", hit=None):
    """Thumbnails of chosen blocks (with some context), framed green (hit) or red (not a hit)."""
    idx = list(idx)
    n = len(idx)
    cols = min(n, 8)
    rws = int(np.ceil(n / cols)) if n else 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        im = task.view(u.scene, key, stretch)
        H, W = im.shape[:2]
        cy, cx = u.y0 + u["size"] // 2, u.x0 + u["size"] // 2
        y0 = int(np.clip(cy - ctx // 2, 0, max(H - ctx, 0)))
        x0 = int(np.clip(cx - ctx // 2, 0, max(W - ctx, 0)))
        ax.imshow(im[y0:y0 + ctx, x0:x0 + ctx])
        _overlay(ax, _outline(task.arr(u.scene, "label") == 1)[y0:y0 + ctx, x0:x0 + ctx], "black")
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((u.x0 - x0 - .5, u.y0 - y0 - .5), u["size"], u["size"], fill=False,
                               ec=HIT_C if good else MISS_C, lw=2.5))
        ax.set_title(f"true {u.truth:.0%}", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------------------- land cover (Hanoi)

LC_NAMES = ["building", "road", "water", "tree", "low vegetation", "cropland", "bare soil or sand"]
FARM_BUILDINGS = "farm_buildings"   # a target for lc_units: buildings in cells that are mostly cropland
LC_COLOURS = np.array([[215, 48, 39], [99, 99, 99], [43, 131, 186], [26, 120, 50], [166, 217, 106],
                       [240, 210, 80], [191, 129, 45], [255, 255, 255]], np.uint8) / 255.0


def lc_metric_table(task):
    """IoU per class and mean IoU per input, pooled over all tiles (pixel by pixel)."""
    cm = {inp: np.zeros((7, 7), np.int64) for inp in task.inputs}
    for sid in task.scenes.index:
        v = task.arr(sid, "valid")
        t = task.arr(sid, "label")[v].astype(np.int64)
        for inp in task.inputs:
            p = task.arr(sid, f"pred_{inp}")[v].astype(np.int64)
            cm[inp] += np.bincount(t * 7 + p, minlength=49).reshape(7, 7)
    paper = {r["input"]: r.get("paper") for r in task.meta["metric_table"]}
    rows = []
    for inp, c in cm.items():
        iou = np.diag(c) / np.maximum(c.sum(0) + c.sum(1) - np.diag(c), 1)
        rows.append({"input": task.labels[inp], **{n: round(v, 2) for n, v in zip(LC_NAMES, iou)},
                     "mean (mIoU)": round(iou.mean(), 3), "paper mIoU": paper.get(inp)})
    return pd.DataFrame(rows).set_index("input")


def lc_classes(target):
    """The class numbers behind a target (a class, several classes or FARM_BUILDINGS)."""
    return (0,) if isinstance(target, str) else tuple(int(t) for t in np.atleast_1d(target))


def lc_units(task, cell_m=80, target=6, min_valid=0.8, farm_share=0.5):
    """Cut every tile into square cells of cell_m metres. For each cell: the true share of the
    target class(es) and, per input, the share the task model labels as that class. Only
    labelled pixels count; cells less than min_valid labelled are left out.
    target=FARM_BUILDINGS: the share of buildings, counted only in cells that are at least
    farm_share cropland (0 elsewhere), on the expert map and on each model's own map."""
    c = max(4, int(round(cell_m / task.px)))
    farm = isinstance(target, str)
    cls = lc_classes(target)
    rows, preds = [], {inp: [] for inp in task.inputs}
    for sid in task.scenes.index:
        lab, val = task.arr(sid, "label"), task.arr(sid, "valid")
        H, W = lab.shape
        ny, nx = H // c, W // c

        def blocks(a):
            return a[:ny * c, :nx * c].reshape(ny, c, nx, c).sum((1, 3)).ravel()
        nv = blocks(val.astype(np.float32))

        def share(classmap):
            s = blocks((np.isin(classmap, cls) & val).astype(np.float32)) / np.maximum(nv, 1)
            if farm:
                crop = blocks(((classmap == 5) & val).astype(np.float32)) / np.maximum(nv, 1)
                s = np.where(crop >= farm_share, s, 0.0)
            return s
        truth = share(lab)
        for inp in task.inputs:
            preds[inp].append(share(task.arr(sid, f"pred_{inp}")))
        for i in range(ny * nx):
            rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": (i // nx) * c,
                         "x0": (i % nx) * c, "size": c, "valid": nv[i] / c / c,
                         "truth": truth[i]})
    units = pd.DataFrame(rows)
    keep = units["valid"].values >= min_valid
    units = units[keep].reset_index(drop=True)
    preds = {k: np.concatenate(v)[keep][None, :] for k, v in preds.items()}
    print(f"{len(units):,} cells of {c * task.px:.0f} m x {c * task.px:.0f} m in "
          f"{units.scene.nunique()} tiles (cells less than {min_valid:.0%} labelled are left out)")
    return units, preds


def _lc_rgb(lab):
    return LC_COLOURS[np.where(lab == 255, 7, lab)]


def lc_show(task, sid, inputs=None, stretch="auto", target=None, box=None, title=None):
    """Top row: the images. Bottom row: the land cover the task model draws on each input; the
    last column is the expert map (white = not labelled). target: outline one class in black."""
    inputs = inputs or ["s2_10m"] + task.inputs
    cols = inputs + ["reference"]
    lab = task.arr(sid, "label")
    edge = _outline(np.isin(lab, lc_classes(target))) if target is not None else None
    fig, axes = plt.subplots(2, len(cols), figsize=(2.5 * len(cols), 5.5), squeeze=False)
    for j, key in enumerate(cols):
        ax, bx = axes[0, j], axes[1, j]
        if key == "reference":
            ax.axis("off")
            bx.imshow(_lc_rgb(lab), interpolation="nearest")
            bx.set_title("expert map")
        else:
            im = task.view(sid, key, stretch)
            if im is None:
                _blank(ax, "image not\ndownloaded")
            else:
                ax.imshow(im, interpolation="nearest")
            ax.set_title(SHORT.get(key, key))
            if key == "s2_10m":
                _blank(bx, "(the model sees the\nupsampled version)")
            else:
                bx.imshow(_lc_rgb(task.arr(sid, f"pred_{key}")), interpolation="nearest")
        if edge is not None and key != "s2_10m":
            _overlay(bx, edge, "black")
        if box:
            for b in (ax, bx):
                b.add_patch(Rectangle((box[1] - .5, box[0] - .5), box[2], box[2], fill=False,
                                      ec="#d73027", lw=2))
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    axes[1, 0].set_ylabel("land cover drawn")
    s = task.scenes.loc[sid]
    fig.suptitle(title or f"{sid}  ({s.group}, 640 m across)", x=0.01, ha="left")
    fig.legend(handles=[Patch(color=LC_COLOURS[i], label=n) for i, n in enumerate(LC_NAMES)]
               + [Patch(fc="white", ec="#999", label="not labelled")],
               loc="lower center", ncol=8, frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()


def lc_gallery(task, units, idx, key="stage2", stretch="auto", ctx=96, title="", hit=None,
               target=6):
    """Thumbnails of chosen cells (with some context), the target class outlined in black from
    the expert map, framed green (hit) or red (not a hit)."""
    idx = list(idx)
    cols = min(len(idx), 8) or 1
    rws = int(np.ceil(len(idx) / cols)) or 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        im = task.view(u.scene, key, stretch)
        H, W = im.shape[:2]
        ctx_ = max(ctx, int(u["size"]) + 16)
        cy, cx = u.y0 + u["size"] // 2, u.x0 + u["size"] // 2
        y0 = int(np.clip(cy - ctx_ // 2, 0, max(H - ctx_, 0)))
        x0 = int(np.clip(cx - ctx_ // 2, 0, max(W - ctx_, 0)))
        ax.imshow(im[y0:y0 + ctx_, x0:x0 + ctx_])
        lab = task.arr(u.scene, "label")[y0:y0 + ctx_, x0:x0 + ctx_]
        _overlay(ax, _outline(np.isin(lab, lc_classes(target))), "black")
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((u.x0 - x0 - .5, u.y0 - y0 - .5), u["size"], u["size"], fill=False,
                               ec=HIT_C if good else MISS_C, lw=2.5))
        ax.set_title(f"true {u.truth:.0%}", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------------------- field boundaries

def ai4_units(task, cell_m=500, small_m2=None, min_px=16, cache=True):
    """Cut every tile into square cells of cell_m metres and count, per cell, the fields whose
    centre lies in it: reference fields and the fields each model draws (one per seed).
    small_m2: count only fields smaller than this (None = all fields)."""
    c = max(8, int(round(cell_m / task.px)))
    key = f"units_{c}_{small_m2}_{min_px}.npz"
    cpath = task.root / key
    seeds = range(len(task.meta.get("models", [0, 1, 2])))
    if cache and cpath.exists():
        z = np.load(cpath, allow_pickle=True)
        units = pd.DataFrame(z["units"].tolist())
        return units, {k[5:]: z[k] for k in z.files if k.startswith("pred_")}
    a_px = task.px * task.px
    rows, preds = [], {inp: [] for inp in task.inputs}
    t0 = time.time()
    for sid in task.scenes.index:
        H, W = int(task.scenes.loc[sid, "h"]), int(task.scenes.loc[sid, "w"])
        ny, nx = H // c, W // c
        yy, xx = np.mgrid[0:H, 0:W]

        def counts(ids, min_px_):
            ids = ids.ravel().astype(np.int64)
            n = int(ids.max()) + 1
            cnt = np.bincount(ids, minlength=n)
            cy = np.bincount(ids, weights=yy.ravel(), minlength=n) / np.maximum(cnt, 1)
            cx = np.bincount(ids, weights=xx.ravel(), minlength=n) / np.maximum(cnt, 1)
            ok = (cnt >= min_px_) & (np.arange(n) > 0)
            if small_m2:
                ok &= cnt * a_px < small_m2
            iy, ix = (cy[ok] // c).astype(int), (cx[ok] // c).astype(int)
            inside = (iy < ny) & (ix < nx)
            return np.bincount(iy[inside] * nx + ix[inside], minlength=ny * nx)

        valid = task.arr(sid, "valid")
        vfrac = valid[:ny * c, :nx * c].reshape(ny, c, nx, c).mean((1, 3)).ravel()
        ref = counts(task.arr(sid, "fields_ref"), min_px)
        for inp in task.inputs:
            preds[inp].append(np.stack([counts(task.arr(sid, f"fields_{inp}_s{s}"), min_px) for s in seeds]))
        for i in range(ny * nx):
            rows.append({"scene": sid, "group": task.scenes.loc[sid, "group"], "y0": (i // nx) * c,
                         "x0": (i % nx) * c, "size": c, "valid": vfrac[i], "truth": int(ref[i])})
        print(f"\r  {sid} done ({time.time() - t0:.0f} s)   ", end="")
    print()
    units = pd.DataFrame(rows)
    preds = {k: np.concatenate(v, axis=1).astype(float) for k, v in preds.items()}
    if cache:
        np.savez(cpath, units=np.array(units.to_dict("records"), dtype=object),
                 **{f"pred_{k}": v for k, v in preds.items()})
    return units, preds


def ai4_tile_counts(task, small_m2=None, min_px=16):
    """Number of fields per tile: reference against each input (mean of the seeds)."""
    a_px = task.px * task.px
    seeds = range(len(task.meta.get("models", [0, 1, 2])))
    rows = []
    for sid in task.scenes.index:
        def n_fields(ids):
            cnt = np.bincount(ids.ravel().astype(np.int64))
            ok = (cnt >= min_px) & (np.arange(len(cnt)) > 0)
            if small_m2:
                ok &= cnt * a_px < small_m2
            return int(ok.sum())
        r = {"tile": sid, "reference": n_fields(task.arr(sid, "fields_ref"))}
        for inp in task.inputs:
            r[task.labels[inp]] = round(np.mean([n_fields(task.arr(sid, f"fields_{inp}_s{s}")) for s in seeds]))
        rows.append(r)
    df = pd.DataFrame(rows).set_index("tile")
    tot = df.sum()
    df.loc["ALL TILES"] = tot
    return df


def ai4_show(task, sid, y0=0, x0=0, size=400, inputs=None, stretch="auto", seed=0, title=None):
    """Top row: the images with the reference field outlines (red). Bottom row: the fields each
    model draws (seed 0), outlined, over its boundary probability."""
    inputs = inputs or ["s2_10m"] + task.inputs
    sl = (slice(y0, y0 + size), slice(x0, x0 + size))
    ref = task.arr(sid, "fields_ref")[sl]
    ref_edge = _boundaries(ref)
    fig, axes = plt.subplots(2, len(inputs), figsize=(3.1 * len(inputs), 6.4), squeeze=False)
    for j, key in enumerate(inputs):
        ax, bx = axes[0, j], axes[1, j]
        im = task.view(sid, key, stretch)
        ax.imshow(im[sl], interpolation="nearest")
        _overlay(ax, ref_edge, "#d73027", 0.9)
        ax.set_title(SHORT.get(key, key))
        if key == "s2_10m":
            _blank(bx, "(each kind of image has\nits own trained model)")
        else:
            prob = task.arr(sid, f"prob_{key}_s0")[sl] / 255.0 if f"prob_{key}_s0" in task.keys(sid) else None
            if prob is not None:
                bx.imshow(prob, cmap="gray_r", vmin=0, vmax=2.5)  # light grey: boundary probability
            f = task.arr(sid, f"fields_{key}_s{seed}")[sl]
            _overlay(bx, _boundaries(f), "#1f8a70", 1)
            n = len(np.unique(f[f > 0]))
            bx.set_title(f"{n} fields drawn here", fontsize=9)
        for b in (ax, bx):
            b.set_xticks([]), b.set_yticks([])
    nref = len(np.unique(ref[ref > 0]))
    axes[1, 0].set_ylabel("model output")
    fig.suptitle(title or f"{sid}: {size * task.px / 1000:.1f} km window, {nref} reference fields",
                 x=0.01, ha="left")
    fig.legend(handles=[Patch(color="#d73027", label="reference field outlines"),
                        Patch(color="#1f8a70", label="fields the model draws")],
               loc="lower center", ncol=2, frameon=False)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


def _boundaries(ids):
    e = np.zeros(ids.shape, bool)
    e[1:, :] |= ids[1:, :] != ids[:-1, :]
    e[:, 1:] |= ids[:, 1:] != ids[:, :-1]
    return e


def ai4_gallery(task, units, idx, key="stage1", stretch="auto", title="", hit=None):
    idx = list(idx)
    cols = min(len(idx), 8) or 1
    rws = int(np.ceil(len(idx) / cols)) or 1
    fig, axes = plt.subplots(rws, cols, figsize=(1.9 * cols, 2.1 * rws), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, idx):
        u = units.iloc[i]
        sl = (slice(u.y0, u.y0 + u["size"]), slice(u.x0, u.x0 + u["size"]))
        ax.imshow(task.view(u.scene, key, stretch)[sl])
        _overlay(ax, _boundaries(task.arr(u.scene, "fields_ref")[sl]), "#d73027", 0.8)
        good = bool(hit[i]) if hit is not None else None
        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False,
                               ec=HIT_C if good else MISS_C, lw=5))
        ax.set_title(f"{u.truth} fields", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


# ---- the notebook's controls (ipywidgets): menus, sliders and buttons that act at once ------
# Pasted after hk.py by build.py. Each part's cell is run once (Runtime > Run all does it);
# its controls redraw at once. Parts 3b to 6 stay closed until the bet of part 3 is revealed,
# then follow the decision settings of part 3. Test by setting widget values in UI.
import contextlib
import html as _html
import importlib
import io
import subprocess

import ipywidgets as W
from IPython.display import display

ANSWERS = {}     # your team's answers; saved as soon as you type or click
UI = {}          # the controls of each part (used for testing)
S = {}           # this session: the task, the brief and the current decision
REFRESH = {}     # what to redraw when the decision settings of part 3 change (by part)
ANSWER_REFRESH = {}   # what to redraw when an answer changes
_WIDE = W.Layout(width="600px")
_LABEL = {"description_width": "200px"}
_CARD = ("border:1px solid #c9c9c9; border-left:6px solid #007DB7; background:#fafafa; "
         "color:#222; padding:10px 16px; max-width:860px; line-height:1.5")
LOCKED = "&#128274; Opens once you reveal the result in part 3."


def _buttons(width):
    return {"button_width": width, "description_width": "200px"}


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _note(text, colour="#333"):
    return f"<div style='color:{colour}; line-height:1.5'>{text}</div>"


def _pre(text):
    return ("<pre style='margin:4px 0; line-height:1.35; font-family:monospace; "
            f"white-space:pre-wrap'>{_html.escape(text)}</pre>")


def _error(exc):
    return _note(f"Something went wrong: {_html.escape(str(exc))}. Run the cell again, or ask "
                 "a facilitator.", "#b00020")


def _run(fn, *args, **kwargs):
    """Call fn with plt.show switched off. Returns (its result, PNGs of the figures it drew,
    what it printed), so the figures can go into image widgets."""
    plt.close("all")
    show, plt.show = plt.show, (lambda *a, **k: None)
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            out = fn(*args, **kwargs)
    finally:
        plt.show = show
    pngs = [_png(plt.figure(n)) for n in list(plt.get_fignums())]
    plt.close("all")
    return out, pngs, buf.getvalue().strip()


def _images(pngs):
    return [W.Image(value=p, format="png", layout=W.Layout(max_width="100%")) for p in pngs]


def _html_table(t):
    try:
        return t.to_html()
    except TypeError:
        return t.to_html(float_format="%.2f")


def _short(task):
    return {k: SHORT[k] for k in task.inputs}


def _kind(task):
    return task.meta["task"]


def _need_decision():
    if "res" not in S:
        display(W.HTML(_note("<b>Run the cell of part 3 first</b>, then run this cell again.",
                             "#b00020")))
        return False
    return True


def _gated(name, build):
    """A part that stays closed until the result of part 3 is revealed. build() makes the
    part once and returns (its widgets, a function that redraws it); afterwards the part
    follows every change of the decision settings."""
    holder = W.VBox([W.HTML(_note(LOCKED, "#8a6d00"))])
    state = {}

    def refresh():
        if not S.get("revealed"):
            return
        try:
            if "redraw" not in state:
                children, state["redraw"] = build()
                holder.children = tuple(children)
            else:
                state["redraw"]()
        except Exception as exc:  # say what went wrong instead of failing silently
            holder.children = (W.HTML(_error(exc)),)
    REFRESH[name] = refresh
    UI[name + "_holder"] = holder
    display(holder)
    refresh()


def _num(x):
    """6 for 6.04, 6.1 for 6.12: numbers as people say them."""
    return f"{x:,.0f}" if abs(x - round(x)) < 0.05 else f"{x:,.1f}"


# ---- answers that save themselves ------------------------------------------------------------

def ask(*questions, fill=None, show=True):
    """Each question is (key, text, kind[, example]): kind is "text", "checks" (tick any of
    the examples, a list) or a list of choices (pick one). The answer is saved in ANSWERS the
    moment it changes; nothing to run. ``fill`` adds a button that copies the examples into
    the empty fields."""
    rows, fields = [], {}
    for q in questions:
        key, text, kind = q[:3]
        example = q[3] if len(q) > 3 else ""
        saved = W.HTML()
        if kind == "checks":
            boxes = [W.Checkbox(value=c in ANSWERS.get(key, ()), description=c, indent=False,
                                layout=W.Layout(width="760px")) for c in example]

            def on_tick(_, key=key, boxes=boxes, saved=saved):
                ANSWERS[key] = [b.description for b in boxes if b.value]
                saved.value = _note("saved ✓", "#1a7f37") if ANSWERS[key] else ""
                for redraw in list(ANSWER_REFRESH.values()):
                    redraw()
            for b in boxes:
                b.observe(on_tick, names="value")
            field = W.VBox(boxes)
            field.boxes = boxes
            fields[key] = field
            rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), field, saved],
                               layout=W.Layout(margin="4px 0 8px 0")))
            continue
        if kind == "text":
            field = W.Text(value=ANSWERS.get(key, ""), continuous_update=True, layout=_WIDE,
                           placeholder=f"e.g. {example}" if example else "type here")
        else:
            field = W.ToggleButtons(options=list(kind), value=ANSWERS.get(key),
                                    style={"button_width": "auto"})

        def on_change(change, key=key, saved=saved):
            ANSWERS[key] = change["new"]
            saved.value = _note("saved ✓", "#1a7f37") if change["new"] not in ("", None) else ""
            for redraw in list(ANSWER_REFRESH.values()):
                redraw()
        field.observe(on_change, names="value")
        field.example = example
        fields[key] = field
        rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), W.HBox([field, saved])],
                           layout=W.Layout(margin="4px 0 8px 0")))
    if fill:
        button = W.Button(description=fill, icon="clipboard", layout=W.Layout(width="300px"))

        def on_fill(_):
            for f in fields.values():
                if isinstance(f, W.Text) and not f.value.strip():
                    f.value = f.example
        button.on_click(on_fill)
        rows.insert(0, button)
    box = W.VBox(rows, layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                       margin="8px 0"))
    box.fields = fields
    if show:
        display(box)
    return box


# ---- part 3: look around ---------------------------------------------------------------------

def _stretch_buttons():
    return W.ToggleButtons(options=[("auto contrast", "auto"), ("as delivered", "none")],
                           value="auto", description="Colours", style=_buttons("110px"),
                           tooltips=["each image stretched for viewing",
                                     "the colours the package stores"])


def explore(task):
    """Menus and sliders to move through the places; the picture follows at once."""
    kind = _kind(task)
    image = W.VBox([])
    status = W.HTML()
    stretch = _stretch_buttons()
    if kind == "ifs_manila":
        sc = task.scenes.sort_values("informal_share", ascending=False)
        tile = W.Dropdown(options=[(f"{sid}  ({s:.0%} informal)", sid)
                                   for sid, s in sc.informal_share.items()],
                          description="Tile (most informal first)", layout=_WIDE, style=_LABEL)
        n_models = len(task.meta.get("models", [])) or 6
        model = W.Dropdown(options=[("mean of the six models", None)]
                           + [(f"model {i + 1} only", i) for i in range(n_models)],
                           description="Model output", style=_LABEL)
        thr = W.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, continuous_update=False,
                            readout_format=".2f", description="Flag a pixel above",
                            layout=_WIDE, style=_LABEL)
        controls = [tile, model, thr, stretch]

        def render():
            ifs_show(task, tile.value, model=model.value, thr=thr.value, stretch=stretch.value)
    elif kind == "hanoi_landcover":
        sort_by = W.Dropdown(options=[(n, i) for i, n in enumerate(LC_NAMES)], value=6,
                             description="Sort tiles by", layout=_WIDE, style=_LABEL)
        tile = W.Dropdown(description="Tile (most first)", layout=_WIDE, style=_LABEL)
        outline = W.Checkbox(value=True, description="Outline that class (expert map) in black",
                             layout=_WIDE, style=_LABEL)
        controls = [sort_by, tile, outline, stretch]

        def tiles(_=None):
            col = f"share_{sort_by.value}"
            sc = task.scenes.sort_values(col, ascending=False)
            tile.options = [(f"{sid}  ({v:.0%} {LC_NAMES[sort_by.value]})", sid)
                            for sid, v in sc[col].items()]
        sort_by.observe(tiles, names="value")
        tiles()

        def render():
            lc_show(task, tile.value, stretch=stretch.value,
                    target=sort_by.value if outline.value else None)
    else:
        area = W.Dropdown(options=[(f"{sid}  ({int(s.n_fields):,} fields)", sid)
                                   for sid, s in task.scenes.iterrows()],
                          value="16_cambodia", description="Area", layout=_WIDE, style=_LABEL)
        size = W.ToggleButtons(options=[("250 m", 100), ("500 m", 200), ("1 km", 400),
                                        ("2 km", 800)], value=400, description="Window",
                               style=_buttons("80px"))
        east = W.IntSlider(value=2000, min=0, max=4000, step=100, continuous_update=False,
                           description="← west | east → (m)", layout=_WIDE, style=_LABEL)
        south = W.IntSlider(value=2000, min=0, max=4000, step=100, continuous_update=False,
                            description="↑ north | south ↓ (m)", layout=_WIDE, style=_LABEL)
        seed = W.ToggleButtons(options=[("model 1", 0), ("model 2", 1), ("model 3", 2)], value=0,
                               description="Trained model (seed)", style=_buttons("80px"))
        controls = [area, size, east, south, seed, stretch]

        def limits(_=None):
            s = task.scenes.loc[area.value]
            east.max = int((int(s.w) - size.value) * task.px) // 100 * 100
            south.max = int((int(s.h) - size.value) * task.px) // 100 * 100
        area.observe(limits, names="value")
        size.observe(limits, names="value")
        limits()

        def render():
            y0, x0 = int(south.value / task.px), int(east.value / task.px)
            ai4_show(task, area.value, y0=y0, x0=x0, size=size.value, seed=seed.value,
                     stretch=stretch.value)

    def draw(_=None):
        try:
            _, pngs, _ = _run(render)
            image.children = _images(pngs)
        except Exception as exc:  # say what went wrong instead of failing silently
            status.value = _error(exc)
    for w in controls:
        w.observe(draw, names="value")
    ui = W.VBox([*controls, status, image])
    ui.controls = controls
    UI["explore"] = ui
    display(ui)
    draw()


# ---- part 1: the brief -----------------------------------------------------------------------
# Mock policy briefs: invented for the workshop. Each sets the scenario text of part 1, the
# decision settings of part 3 and the area and rhythm of part 5.

SCENARIO_FIELDS = [("team", "Team name"),
                   ("decision_maker", "Who decides?"),
                   ("decision", "What do they decide?"),
                   ("budget", "Budget: how many places can they act on?"),
                   ("hit", "What counts as a hit (a place worth the visit)?"),
                   ("cost_of_wasted_visit", "What does a wasted visit cost?"),
                   ("cost_of_missed_place", "What does a missed place cost?")]

BRIEFS = {
    "ifs_manila": [
        dict(title="Where should the validation teams go first?",
             frm="Regional Director, housing and urban development office",
             to="Planning and monitoring team (you)",
             situation="The city is updating its Local Shelter Plan. Its list of informal "
                       "settlements is years old, and families missing from the list cannot be "
                       "planned for. Your teams can validate 20 blocks this quarter. There is "
                       "no budget for new commercial images of the whole city every year.",
             question="Can free Sentinel-2 images, sharpened by super-resolution, show us "
                      "which 20 blocks to validate first? Or do we have to buy real "
                      "high-resolution images?",
             scenario=dict(
                 decision_maker="Regional office for housing and urban development, settlement "
                                "upgrading programme",
                 decision="Which city blocks get a field validation this quarter",
                 budget="20 validation visits, one 100 m x 100 m block each",
                 hit="a block where at least a quarter of the area is informal settlement",
                 cost_of_wasted_visit="1 day for a team of 2",
                 cost_of_missed_place="families left out of the shelter plan for a year: about "
                                      "3 survey days' worth"),
             settings=dict(budget=20, size=100, hit_at=0.25, cw=1, cm=3),
             area=(620, "Metro Manila is about 620 km²"), updates=4),
        dict(title="A settlement watch list for local governments?",
             frm="Deputy Director General, national space agency",
             to="Space applications team (you)",
             situation="Local governments ask the agency where informal settlements are "
                       "likely, so they can plan services. From free images, the agency could "
                       "send every city a fresh list each quarter. But local staff are busy: a "
                       "city will check only the top 50 blocks, and every wrong flag on a "
                       "formal neighbourhood costs the agency trust.",
             question="Is a list made from super-resolved images good enough to send out as "
                      "'look here first'? Or should we wait until we can pay for real "
                      "high-resolution images?",
             scenario=dict(
                 decision_maker="National space agency, space applications team",
                 decision="Whether to send local governments a quarterly list of blocks that "
                          "likely hold informal settlements",
                 budget="the 50 blocks at the top of a city's list, 100 m x 100 m each",
                 hit="a block where at least 10 % of the area is informal settlement",
                 cost_of_wasted_visit="a wrong flag costs trust and local staff time: 2",
                 cost_of_missed_place="a settlement that stays off the list this quarter: 1"),
             settings=dict(budget=50, size=100, hit_at=0.10, cw=2, cm=1),
             area=(300000, "the Philippines is about 300,000 km²"), updates=4),
    ],
    "hanoi_landcover": [
        dict(title="Which building sites get a dust inspection?",
             frm="Head, city environment department",
             to="Air-quality enforcement unit (you)",
             situation="In the dry season, dust from building sites, cleared land and sand "
                       "yards is a main source of the city's air pollution. Your inspectors "
                       "can make 20 inspections a month. Today they go where complaints come "
                       "from.",
             question="Can super-resolved Sentinel-2 images point the inspectors to open, bare "
                      "ground each month? Or would free Sentinel-2 do?",
             scenario=dict(
                 decision_maker="City environment department, air-quality enforcement unit",
                 decision="Which 80 m x 80 m cells get a dust-control inspection this month "
                          "(open construction sites, cleared land, sand yards)",
                 budget="20 inspections a month",
                 hit="a cell where at least 20 % of the ground is bare soil or sand",
                 cost_of_wasted_visit="1 inspector-day",
                 cost_of_missed_place="a dust source left uncontrolled for a month: about 3 "
                                      "inspector-days' worth"),
             settings=dict(target=6, size=80, budget=20, hit_at=0.20, cw=1, cm=3),
             area=(3360, "the city of Hanoi is about 3,360 km²"), updates=12),
        dict(title="Are houses going up on protected farmland?",
             frm="Director, provincial land-use and zoning office",
             to="Zoning compliance team (you)",
             situation="Rice land at the city's edge is protected: building on it needs a "
                       "conversion permit. Buildings appear anyway, a few at a time. Your team "
                       "can make 15 site checks a quarter and has no recent map.",
             question="Can super-resolved images show where buildings stand in farmland, so "
                      "the team knows where to check? (A real system would compare two dates. "
                      "Here you test the first step: finding buildings among fields.)",
             scenario=dict(
                 decision_maker="Provincial land-use and zoning office, zoning compliance team",
                 decision="Which 160 m x 160 m cells of farmland get a site check for buildings "
                          "without a permit",
                 budget="15 site checks a quarter",
                 hit="a cell that is at least half cropland and where buildings cover at least "
                     "5 % of the ground",
                 cost_of_wasted_visit="1 inspector-day",
                 cost_of_missed_place="a building without a permit that stands for another "
                                      "quarter: about 3 inspector-days' worth"),
             settings=dict(target=FARM_BUILDINGS, size=160, budget=15, hit_at=0.05, cw=1, cm=3),
             area=(3360, "the city of Hanoi is about 3,360 km²"), updates=4),
        dict(title="Which parts of the road map are out of date?",
             frm="Chief, planning division, public works department",
             to="Road inventory team (you)",
             situation="The road inventory decides which farm-to-market roads get maintenance "
                       "money, and many small roads are missing from it. Your mappers can "
                       "redraw 40 map cells a month. They should start where there is most "
                       "road to draw.",
             question="Can super-resolved images show the mappers which cells hold the most "
                      "road? Many roads are narrower than a Sentinel-2 pixel.",
             scenario=dict(
                 decision_maker="Public works department, planning division",
                 decision="Which 160 m x 160 m cells the mappers redraw first",
                 budget="40 cells a month",
                 hit="a cell where roads cover at least 10 % of the ground",
                 cost_of_wasted_visit="half a mapper-day",
                 cost_of_missed_place="a road that stays off the inventory, and off the "
                                      "maintenance list, for a year: about 2 mapper-days' worth"),
             settings=dict(target=1, size=160, budget=40, hit_at=0.10, cw=0.5, cm=2),
             area=(3360, "the city of Hanoi is about 3,360 km²"), updates=1),
        dict(title="Where is the tree cover we must protect?",
             frm="Head, parks and green space office",
             to="Urban greening team (you)",
             situation="The city's heat action plan protects its last large groups of trees. "
                       "The office wants to survey the 20 greenest cells this year and declare "
                       "them protected.",
             question="Do we need super-resolution for this, or is free Sentinel-2 enough?",
             scenario=dict(
                 decision_maker="Parks and green space office, urban greening team",
                 decision="Which 80 m x 80 m cells get a tree survey and a protection order",
                 budget="20 surveys this year",
                 hit="a cell where trees cover at least 30 % of the ground",
                 cost_of_wasted_visit="1 surveyor-day",
                 cost_of_missed_place="a grove that stays unprotected: about 1 surveyor-day's "
                                      "worth"),
             settings=dict(target=3, size=80, budget=20, hit_at=0.30, cw=1, cm=1),
             area=(3360, "the city of Hanoi is about 3,360 km²"), updates=1),
    ],
    "ai4_fields": [
        dict(title="Which villages get the farmer-registry teams?",
             frm="Regional Executive Director, department of agriculture",
             to="Smallholder support programme (you)",
             situation="Seeds, fertiliser vouchers and crop insurance reach only registered "
                       "farmers, and the smallest farms are most often missing from the "
                       "registry. Your extension workers can visit 30 areas this season to "
                       "enrol farmers.",
             question="Can super-resolved images show where the small farms are, so the teams "
                      "go there first? Or would free Sentinel-2 do?",
             scenario=dict(
                 decision_maker="Department of agriculture regional office, smallholder support "
                                "programme",
                 decision="Which 500 m x 500 m squares get an extension-worker visit to enrol "
                          "small farms",
                 budget="30 visits this season",
                 hit="a square with at least 30 fields smaller than 0.25 ha",
                 cost_of_wasted_visit="1 extension-worker day",
                 cost_of_missed_place="about 30 smallholders not enrolled this season: about 2 "
                                      "worker-days' worth"),
             settings=dict(budget=30, size=500, small=2500, hit_at=30, cw=1, cm=2),
             area=(5000, "a large province is about 5,000 km²"), updates=2),
        dict(title="Where does the farm census need a second enumerator?",
             frm="Chief, agricultural statistics division, national statistics office",
             to="Census planning team (you)",
             situation="In the census of agriculture, one enumerator covers one square "
                       "kilometre. Squares with very many parcels need a second enumerator, "
                       "and the office can pay for 20. The last parcel count is ten years old.",
             question="Can satellite images count parcels well enough to choose those 20 "
                      "squares? Does super-resolution add anything to free Sentinel-2?",
             scenario=dict(
                 decision_maker="National statistics office, agricultural statistics division",
                 decision="Which 1 km squares get a second enumerator",
                 budget="20 extra enumerators",
                 hit="a square with at least 200 parcels (fields of any size)",
                 cost_of_wasted_visit="an enumerator sent where one was enough: 1",
                 cost_of_missed_place="a square counted late or badly: 1"),
             settings=dict(budget=20, size=1000, small=0, hit_at=200, cw=1, cm=1),
             area=(5000, "a large province is about 5,000 km²"), updates=1),
    ],
}
OWN_DECISION = "Our own decision (we write the brief)"


def _memo(b):
    return (f"<div style='{_CARD}'>"
            "<div style='color:#8a6d00; font-size:12px; letter-spacing:.5px'>MOCK BRIEF · "
            "invented for this workshop. The kind of decision is real; the office and the "
            "numbers are not.</div>"
            f"<div style='margin-top:6px'><b>From:</b> {b['frm']}<br><b>To:</b> {b['to']}<br>"
            f"<b>Subject:</b> {b['title']}</div>"
            f"<p style='margin:8px 0'>{b['situation']}</p>"
            f"<p style='margin:8px 0'><b>What I need from you:</b> {b['question']}</p>"
            "<p style='margin:8px 0 0 0'><b>Reply on one slide:</b> deploy, pilot or don't "
            "use; the numbers; your safeguards.</p></div>")


_OWN_MEMO = (f"<div style='{_CARD}'><b>Your own brief.</b> Pick a decision that someone in your "
             "agency really makes and that a map could inform. Fill in the fields below, then "
             "set the same budget and hit rule in part 3.</div>")


def mission(task):
    """Part 1: choose a mock brief (or write your own). The choice fills the scenario fields
    below it, the decision settings of part 3 and the area of part 5; everything stays
    editable."""
    kind = _kind(task)
    briefs = BRIEFS[kind]
    S["task"] = task
    choose = W.RadioButtons(options=[(b["title"], i) for i, b in enumerate(briefs)]
                            + [(OWN_DECISION, -1)], value=0, layout=W.Layout(width="100%"))
    memo = W.HTML()
    form = ask(*[(k, t, "text", briefs[0]["scenario"].get(k, "Team A"))
                 for k, t in SCENARIO_FIELDS], show=False)

    def on_choose(_=None):
        b = briefs[choose.value] if choose.value >= 0 else None
        S["brief"] = b
        ANSWERS["brief"] = b["title"] if b else OWN_DECISION
        memo.value = _memo(b) if b else _OWN_MEMO
        for key, field in form.fields.items():
            if key != "team":
                field.value = b["scenario"][key] if b else ""
        for apply in (S.get("apply_settings"), S.get("apply_area")):
            if b and apply:
                apply(b)
        for redraw in list(ANSWER_REFRESH.values()):
            redraw()
    choose.observe(on_choose, names="value")
    on_choose()
    ui = W.VBox([W.HTML("<b>Choose your brief</b>"), choose, memo,
                 W.HTML("<br><b>Your scenario</b> (from the brief; change what does not fit "
                        "your agency)"), form])
    ui.choose, ui.memo, ui.fields = choose, memo, form.fields
    UI["mission"] = UI["scenario"] = ui
    display(ui)


def scenario_dict():
    """The scenario as typed in part 1 (empty fields: '(not answered)')."""
    return {k: (ANSWERS.get(k) or "").strip() or "(not answered)" for k, _ in SCENARIO_FIELDS}


# ---- part 3: the bet and the decision --------------------------------------------------------

def _int_slider(value, lo, hi, step, text):
    return W.IntSlider(value=value, min=lo, max=hi, step=step, continuous_update=False,
                       description=text, layout=_WIDE, style=_LABEL)


def _cost(value, text):
    return W.BoundedFloatText(value=value, min=0, max=10000, step=0.5, description=text,
                              style=_LABEL)


def _list(items):
    """'a, b and c'."""
    items = list(items)
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


def _same_result_more_visits(preds, hit, k, labels):
    """How many visits each input needs to find what the best input finds in k visits."""
    means = {i: np.mean([expected_hits(pm, hit, k) for pm in np.atleast_2d(p)])
             for i, p in preds.items()}
    ref = "hr" if "hr" in preds else max(means, key=means.get)
    if means[ref] < 0.5:
        return "", {}
    need = {}
    for i, p in preds.items():
        if i != ref:
            n = [visits_needed(pm, hit, means[ref]) or len(hit) for pm in np.atleast_2d(p)]
            need[i] = int(np.median(n))
    more = [f"<b>{need[i]:,}</b> with {labels[i]}" for i in sorted(need, key=need.get)
            if need[i] > k]
    if more:
        more[0] = more[0].replace("</b> with", "</b> visits with")
    same = [labels[i] for i in need if need[i] <= k]
    text = (f"<b>Same result, more visits.</b> {labels[ref]} finds {_num(means[ref])} hits in "
            f"{k} visits. ")
    if more:
        text += f"To match it, you need about {_list(more)}. "
    if same:
        text += f"{_list(same)} {'does' if len(same) == 1 else 'do'} just as well."
    return text, {labels[i]: v for i, v in need.items()}


def _reading(res, k, labels, more, bets):
    """The result in plain words, for people who do not read charts every day."""
    items = []
    for n, name in enumerate(labels.values()):
        r = res.loc[name]
        spread = r["range over models"].replace("–", " to ")
        if n == 0:
            spread = f" ({spread}, depending on the model)" if spread else ""
            items.append(f"<b>{name}</b>: <b>{_num(r['hits'])}</b> of your {k} visits are "
                         f"hits{spread}.")
        else:
            items.append(f"<b>{name}</b>: <b>{_num(r['hits'])}</b>"
                         + (f" ({spread})." if spread else "."))
    items.append(f"No map (random picks): {_num(res.loc['random', 'hits'])}. Perfect "
                 f"knowledge: {_num(res.loc['perfect', 'hits'])}.")
    if more:
        items.append(more)
    if bets:
        items.append("<b>Your bets:</b> " + "; ".join(
            f"{n} {b} (result {_num(res.loc[n, 'hits'])})" for n, b in bets.items()) + ".")
    return (f"<div style='{_CARD}'><b>In plain words</b><ul style='margin:4px 0'>"
            + "".join(f"<li>{i}</li>" for i in items) + "</ul></div>")


def decision_panel(task):
    """Part 3: the settings of your decision, your bet and, after the reveal, the result. Every
    later change of a setting recomputes at once, and parts 3b to 6 follow. The places
    (blocks, cells or squares) are computed once per setting and kept."""
    kind = _kind(task)
    labels = _short(task)
    S.update(task=task, cache={}, revealed=False)
    brief = S.get("brief") or BRIEFS[kind][0]
    d = brief["settings"]
    if kind == "ifs_manila":
        budget = _int_slider(d["budget"], 5, 100, 5, "Budget (blocks you visit)")
        size = W.ToggleButtons(options=[("50 m", 50), ("100 m", 100), ("200 m", 200)],
                               value=d["size"], description="Block size", style=_buttons("80px"))
        hit_at = W.FloatSlider(value=d["hit_at"], min=0.05, max=0.9, step=0.05,
                               continuous_update=False, readout_format=".0%", layout=_WIDE,
                               style=_LABEL, description="Hit = informal share at least")
        w = dict(budget=budget, size=size, hit_at=hit_at)
        unit_controls = [size]

        def places():
            return ifs_units(task, cell_m=size.value)

        def settings():
            return {"block": f"{size.value} m", "hit": f"informal share ≥ {hit_at.value:.0%}"}

        def title():
            return f"Visit {budget.value} blocks of {size.value} m: how many are hits?"
        place_word = "blocks"
    elif kind == "hanoi_landcover":
        target = W.Dropdown(options=[(n, i) for i, n in enumerate(LC_NAMES)]
                            + [("built-up (building + road)", (0, 1)),
                               ("green (tree + low vegetation)", (3, 4)),
                               ("buildings in farmland (cells at least half cropland)",
                                FARM_BUILDINGS)],
                            value=d["target"], description="Land cover you look for",
                            layout=_WIDE, style=_LABEL)
        size = W.ToggleButtons(options=[("40 m", 40), ("80 m", 80), ("160 m", 160)],
                               value=d["size"], description="Cell size", style=_buttons("80px"))
        budget = _int_slider(d["budget"], 5, 100, 5, "Budget (cells you visit)")
        hit_at = W.FloatSlider(value=d["hit_at"], min=0.05, max=0.9, step=0.05,
                               continuous_update=False, readout_format=".0%", layout=_WIDE,
                               style=_LABEL, description="Hit = share of the cell at least")
        w = dict(target=target, size=size, budget=budget, hit_at=hit_at)
        unit_controls = [target, size]

        def what():
            name = next(n for n, v in target.options if v == target.value)
            return name.split(" (cells")[0]

        def places():
            return lc_units(task, cell_m=size.value, target=target.value)

        def settings():
            extra = " (in cells at least half cropland)" if target.value == FARM_BUILDINGS else ""
            return {"looking for": what(), "cell": f"{size.value} m",
                    "hit": f"{what().split(' in farmland')[0]} ≥ {hit_at.value:.0%} of the cell"
                           + extra}

        def title():
            return f"Visit {budget.value} cells of {size.value} m ({what()}): how many are hits?"
        place_word = "cells"
    else:
        budget = _int_slider(d["budget"], 5, 100, 5, "Budget (squares you visit)")
        size = W.ToggleButtons(options=[("250 m", 250), ("500 m", 500), ("1 km", 1000)],
                               value=d["size"], description="Square size", style=_buttons("80px"))
        small = W.ToggleButtons(options=[("all fields", 0), ("< 0.1 ha", 1000),
                                         ("< 0.25 ha", 2500), ("< 0.5 ha", 5000)],
                                value=d["small"], description="Count fields",
                                style=_buttons("90px"))
        hit_at = _int_slider(d["hit_at"], 1, 400, 1, "Hit = at least this many fields")
        w = dict(budget=budget, size=size, small=small, hit_at=hit_at)
        unit_controls = [size, small]

        def places():
            return ai4_units(task, cell_m=size.value, small_m2=small.value or None)

        def settings():
            what = "fields" if not small.value else f"fields < {small.value / 10000:g} ha"
            return {"square": f"{size.value} m", "hit": f"≥ {hit_at.value} {what}"}

        def title():
            return f"Visit {budget.value} squares of {size.value} m: how many are hits?"
        place_word = "squares"
    w["cw"], w["cm"] = _cost(d["cw"], "Cost of a wasted visit"), _cost(d["cm"],
                                                                         "Cost of a missed place")
    controls = list(w.values())

    status = W.HTML()
    bets = {k: W.IntSlider(value=0, min=0, max=budget.value, continuous_update=False,
                           description=f"Hits with {labels[k]}", layout=_WIDE, style=_LABEL)
            for k in task.inputs}
    reveal = W.Button(description="Lock in our bet and show the result", icon="unlock",
                      button_style="primary", layout=W.Layout(width="320px"))
    skip = W.Button(description="Show the result without a bet",
                    layout=W.Layout(width="260px"))
    bet_head = W.HTML()
    bet_box = W.VBox([bet_head, *bets.values(), W.HBox([reveal, skip])],
                     layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                     margin="10px 0"))
    chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
    reading = W.HTML()
    table = W.HTML()
    result = W.VBox([])

    def bet_text():
        bet_head.value = (
            f"<b>Your bet.</b> Of your {budget.value} visits, how many will be hits? Agree on "
            "one number per kind of image, then lock it in.")

    def compute(_=None):
        if S.get("hold"):
            return
        key = tuple(c.value for c in unit_controls)
        try:
            for b in bets.values():
                b.max = budget.value
            bet_text()
            if key not in S["cache"]:
                status.value = _note(f"Cutting the map into {place_word} ... (the first time "
                                     "for a setting can take up to 20 seconds)")
                (units, preds), _, said = _run(places)
                S["cache"][key] = (units, preds, said)
            units, preds, said = S["cache"][key]
            hit = units.truth.values >= hit_at.value
            res, _, said2 = _run(decide, units, preds, hit, budget.value, w["cw"].value,
                                 w["cm"].value, labels=labels)
            more, need = _same_result_more_visits(preds, hit, min(budget.value, len(hit)), labels)
            S.update(units=units, preds=preds, hit=hit, res=res, budget=budget.value,
                     target=target.value if kind == "hanoi_landcover" else None,
                     hit_at=hit_at.value, visits_needed=need,
                     cost_wasted=w["cw"].value, cost_missed=w["cm"].value, title=title(),
                     settings={"budget": budget.value, **settings(),
                               "cost of a wasted visit": f"{w['cw'].value:g}",
                               "cost of a missed place": f"{w['cm'].value:g}"},
                     place_word=place_word)
            status.value = _pre("\n".join(x for x in (said, said2) if x))
            if not S["revealed"]:
                return
            guess = {labels[k]: b.value for k, b in bets.items()} if S.get("bets") else None
            _, pngs, _ = _run(plot_decision, res, budget.value, title(), guess)
            chart.value = pngs[0]
            reading.value = _reading(res, budget.value, labels, more, S.get("bets"))
            shown = res if len(np.atleast_2d(next(iter(preds.values())))) > 1 else res.drop(
                columns="range over models")
            table.value = shown.to_html()
            result.children = (chart, reading, table)
        except Exception as exc:  # say what went wrong instead of failing silently
            status.value = _error(exc)
            return
        for redraw in list(REFRESH.values()):
            try:
                redraw()
            except Exception:  # a later part must not break the decision cell
                pass

    def on_reveal(button):
        if button is reveal:
            S["bets"] = {labels[k]: b.value for k, b in bets.items()}
            ANSWERS["bets"] = dict(S["bets"])
        S["revealed"] = True
        for b in bets.values():
            b.disabled = True
        reveal.disabled = skip.disabled = True
        reveal.description = "Bet locked in" if S.get("bets") else "No bet placed"
        compute()
    reveal.on_click(on_reveal)
    skip.on_click(on_reveal)

    def apply_settings(b):
        """Set the controls to a brief's settings (one recomputation at the end)."""
        S["hold"] = True
        try:
            for name, value in b["settings"].items():
                if name in w:
                    w[name].value = value
        finally:
            S["hold"] = False
        compute()
    S["apply_settings"] = apply_settings

    for c in controls:
        c.observe(compute, names="value")
    ui = W.VBox([W.HTML("<b>Settings of your decision</b> (from your brief; change them to "
                        "fit your case). Costs are in your own units, such as staff-days. "
                        "Only the ratio matters."),
                 *controls, status, bet_box, result])
    ui.controls = {c.description: c for c in controls}
    ui.w, ui.bets, ui.reveal, ui.skip, ui.result = w, bets, reveal, skip, result
    UI["decision"] = ui
    display(ui)
    compute()


def current():
    """The current decision of part 3: units (one row per place), preds ({input: array
    (models, places)}), hit (bool per place) and res (the table)."""
    return S["units"], S["preds"], S["hit"], S["res"]


def peek(value):
    """Show a result only once the bet of part 3 is revealed (so Run all spoils nothing)."""
    if S.get("revealed"):
        return value
    print("Reveal the result in part 3 first, then run this cell again.")


# ---- part 3b: other budgets ------------------------------------------------------------------

def sweep_view():
    """Share of hits for budgets from 5 to 100; follows the settings of part 3."""
    if not _need_decision():
        return

    def build():
        chart = W.Image(format="png", layout=W.Layout(max_width="100%"))

        def draw():
            _, pngs, _ = _run(sweep, S["preds"], S["hit"], labels=_short(S["task"]),
                              title=f"Share of visits that are hits ({S['place_word']}, "
                                    f"{S['settings']['hit']})")
            chart.value = pngs[0]
        draw()
        UI["sweep"] = chart
        return [chart], draw
    _gated("sweep", build)


# ---- part 4: which places, who is left out, and the failures ---------------------------------

def _gallery(task, idx, key, title):
    fn = {"ifs_manila": ifs_gallery, "hanoi_landcover": lc_gallery, "ai4_fields": ai4_gallery}[
        _kind(task)]
    if not len(idx):
        return []
    extra = {"target": S["target"]} if _kind(task) == "hanoi_landcover" else {}
    _, pngs, _ = _run(fn, task, S["units"], idx, key=key, hit=S["hit"], title=title, **extra)
    return pngs


def _detail(task, i):
    u = S["units"].iloc[i]
    kind = _kind(task)
    if kind == "ifs_manila":
        return _run(ifs_show, task, u.scene, box=(u.y0, u.x0, u["size"]))[1]
    if kind == "hanoi_landcover":
        return _run(lc_show, task, u.scene, target=S["target"], box=(u.y0, u.x0, u["size"]))[1]
    return _run(ai4_show, task, u.scene, y0=int(u.y0), x0=int(u.x0), size=int(u["size"]))[1]


_WHERE = {"ifs_manila": ("group", "city"), "hanoi_landcover": ("group", "source image"),
          "ai4_fields": ("scene", "area")}


def _where_html(task):
    by, word = _WHERE[_kind(task)]
    t = where_table(S["units"], S["preds"], S["hit"], S["budget"], by=by, labels=_short(task))
    visits = [c for c in t.columns if c.startswith("visits")]
    left_out = [str(a) for a, r in t.iterrows() if r["true hits"] >= 5 and r[visits].max() == 0]
    note = (f"<br><b>Left out by every kind of image:</b> {', '.join(left_out)}."
            if left_out else "")
    return (f"<b>Where would your visits go?</b> One row per {word}: its {S['place_word']}, "
            f"its true hits, and the visits each kind of image sends there (of {S['budget']}). "
            f"Many hits and no visits means the {word} is left out.{note}" + _html_table(t))


def picks_view(max_shown=16):
    """Where the visits go, how much the lists overlap, the places each input would pick, its
    failures and one place in detail. Follows the settings of part 3."""
    if not _need_decision():
        return

    def build():
        task = S["task"]
        labels = _short(task)
        shown = [k for k in task.inputs if task.has_image[k]]
        where = W.HTML()
        overlap = W.HTML()
        pick = W.ToggleButtons(options=[(labels[k], k) for k in task.inputs],
                               value="stage2" if "stage2" in task.inputs else task.inputs[-1],
                               description="Places picked with", style=_buttons("120px"))
        default_img = "stage1" if _kind(task) == "ai4_fields" else "stage2"
        on = W.ToggleButtons(options=[(labels[k], k) for k in shown], value=default_img,
                             description="Shown on the image of", style=_buttons("120px"))
        galleries = W.VBox([])
        which = W.ToggleButtons(options=[("a confident false alarm", "false"),
                                         ("a missed hit", "missed")], value="false",
                                description="Look closer at", style=_buttons("170px"))
        number = W.IntSlider(value=1, min=1, max=8, continuous_update=False, layout=_WIDE,
                             style=_LABEL, description="Number (1 = most striking)")
        detail = W.VBox([])
        status = W.HTML()

        def lists():
            score = np.atleast_2d(S["preds"][pick.value]).mean(0)
            hit, k = S["hit"], S["budget"]
            order = np.argsort(-score)
            chosen = set(order[:k])
            false = [i for i in order if not hit[i]][:8]
            missed = [i for i in np.argsort(-S["units"].truth.values)
                      if hit[i] and i not in chosen][:8]
            return top_k(score, k), false, missed

        def draw_detail(_=None):
            try:
                _, false, missed = lists()
                idx = false if which.value == "false" else missed
                if not idx:
                    detail.children = (W.HTML(_note("None: nothing to show for this setting.")),)
                    return
                number.max = len(idx)
                detail.children = _images(_detail(task, idx[min(number.value, len(idx)) - 1]))
            except Exception as exc:
                status.value = _error(exc)

        def draw(_=None):
            try:
                status.value = ""
                where.value = _where_html(task)
                overlap.value = _html_table(overlap_table(S["preds"], S["budget"], labels=labels))
                picks, false, missed = lists()
                name, word = labels[pick.value], S["place_word"]
                note = (f" (the first {max_shown} of {len(picks)})"
                        if len(picks) > max_shown else "")
                pngs = (_gallery(task, picks[:max_shown], on.value,
                                 f"The {S['budget']} {word} you would visit using {name}{note}")
                        + _gallery(task, false, on.value,
                                   f"{name}: highest-scoring {word} that are NOT hits")
                        + _gallery(task, missed, on.value,
                                   f"Clear hits that {name} would NOT visit"))
                galleries.children = _images(pngs)
                draw_detail()
            except Exception as exc:
                status.value = _error(exc)
        for c in (pick, on):
            c.observe(draw, names="value")
        for c in (which, number):
            c.observe(draw_detail, names="value")
        finds = ask(("RESCUE", "A place where super-resolution changed the decision for the "
                     "better (which, and why)", "text"),
                    ("FAILURE", "A failure worth telling the room about (where, and what went "
                     "wrong)", "text"), show=False)
        ui = W.VBox([where,
                     W.HTML("<br><b>Do the images pick the same places?</b> Share of picks "
                            "that two kinds of image have in common (100 % = the same list)"),
                     overlap,
                     W.HTML("<br><b>The places themselves</b> (green frame = hit, red frame = "
                            "wasted visit)"),
                     pick, on, status, galleries,
                     W.HTML("<br><b>One place in detail</b> (for the kind of image picked above)"),
                     which, number, detail, finds])
        ui.pick, ui.on, ui.which, ui.number, ui.galleries, ui.detail, ui.where = (
            pick, on, which, number, galleries, detail, where)
        UI["picks"], UI["finds"] = ui, finds
        draw()
        return [ui], draw
    _gated("picks", build)


# ---- part 5: is it worth the money? ----------------------------------------------------------

PRICE_NOTE = ("Starting prices: real images about USD 4–19 per km² at list price (Sessions 1 "
              "and 3); super-resolution about USD 0.004–0.013 per km² in computing (Session 4); "
              "Sentinel-2 free. Change them to yours.")


def cost_view():
    """Part 5: what the images would cost for your whole area, and a mixed strategy: screen
    everything with a cheap input, buy real images only where it flags, then pick the visits.
    Follows the settings of part 3."""
    if not _need_decision():
        return

    def build():
        task = S["task"]
        labels = _short(task)
        has_hr = "hr" in task.inputs
        b = S.get("brief") or BRIEFS[_kind(task)][0]
        area = W.BoundedFloatText(value=b["area"][0], min=1, max=1e8, step=10,
                                  description="Area you must watch (km²)", style=_LABEL)
        area_note = W.HTML(_note(f"({b['area'][1]})", "#777"))
        updates = W.ToggleButtons(options=[("once a year", 1), ("each season", 2),
                                           ("every quarter", 4), ("every month", 12)],
                                  value=b["updates"], description="A fresh map",
                                  style=_buttons("110px"))
        price_hr = W.BoundedFloatText(value=10, min=0, max=1000, step=1, style=_LABEL,
                                      description="Real images, USD per km²")
        price_sr = W.BoundedFloatText(value=0.01, min=0, max=100, step=0.005, style=_LABEL,
                                      description="Super-resolution, USD per km²")
        cheap = [k for k in task.inputs if k != "hr"]
        screen = W.ToggleButtons(options=[(labels[k], k) for k in cheap], value=cheap[-1],
                                 description="Screen everything with", style=_buttons("120px"))
        share = W.IntSlider(value=5, min=1, max=30, continuous_update=False, layout=_WIDE,
                            style=_LABEL, description="Buy real images for the top (%)")
        chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
        reading = W.HTML()
        status = W.HTML()

        def apply_area(brief):
            area.value, updates.value = brief["area"][0], brief["updates"]
            area_note.value = _note(f"({brief['area'][1]})", "#777")
        S["apply_area"] = apply_area

        def draw(_=None):
            try:
                status.value = ""
                res, preds, hit, k = S["res"], S["preds"], S["hit"], S["budget"]
                k = min(k, len(hit))
                a = area.value * updates.value
                rows = [dict(name=f"{labels[i]} only", hits=float(res.loc[labels[i], "hits"]),
                             cost=0 if i == "s2" else a * price_sr.value, colour=COLOURS[i])
                        for i in cheap]
                lines = []
                if has_hr:
                    m = max(k, int(round(share.value / 100 * len(hit))))
                    mixed = float(np.mean([screen_then_confirm(ps, ph, hit, k, m) for ps, ph in zip(
                        np.atleast_2d(preds[screen.value]), np.atleast_2d(preds["hr"]))]))
                    bill = a * share.value / 100 * price_hr.value + (
                        0 if screen.value == "s2" else a * price_sr.value)
                    full = float(res.loc[labels["hr"], "hits"])
                    rows.append(dict(name=f"{labels[screen.value]} to screen, real images\n"
                                          f"for the top {share.value} %", hits=mixed, cost=bill,
                                     colour="#f2a900"))
                    rows.append(dict(name="Real images everywhere", hits=full,
                                     cost=a * price_hr.value, colour=COLOURS["hr"]))
                    alone = float(res.loc[labels[screen.value], "hits"])
                    lines.append(
                        f"<b>Screen, then buy.</b> {labels[screen.value]} screens every "
                        f"{S['place_word'][:-1]}. You buy real images for the top "
                        f"{share.value} % and choose your {k} visits there. Result: "
                        f"<b>{_num(mixed)} hits</b> ({labels[screen.value]} alone: "
                        f"{_num(alone)}; real images everywhere: {_num(full)}). Imagery bill: "
                        f"<b>USD {bill:,.0f}</b> a year instead of USD "
                        f"{a * price_hr.value:,.0f}.")
                    S["strategy"] = [
                        f"{labels[screen.value]} screens, real images for the top "
                        f"{share.value} %: {_num(mixed)} hits, USD {bill:,.0f} a year",
                        f"real images everywhere: {_num(full)} hits, USD "
                        f"{a * price_hr.value:,.0f} a year",
                        f"{labels[screen.value]} only: {_num(alone)} hits, USD "
                        f"{0 if screen.value == 's2' else a * price_sr.value:,.0f} a year"]
                else:
                    best = max(cheap, key=lambda i: res.loc[labels[i], "hits"])
                    lines.append(
                        "<b>This dataset has no real high-resolution images</b>, so buying "
                        "them cannot be tested here. For scale: real images of your area "
                        f"would cost about <b>USD {a * price_hr.value:,.0f}</b> a year; "
                        f"super-resolution about USD {a * price_sr.value:,.0f}.")
                    S["strategy"] = [
                        f"{labels[best]}: {_num(res.loc[labels[best], 'hits'])} hits, USD "
                        f"{a * price_sr.value:,.0f} a year",
                        f"Sentinel-2: {_num(res.loc[labels['s2'], 'hits'])} hits, free",
                        f"real images (not tested): USD {a * price_hr.value:,.0f} a year"]
                lines.append("<span style='color:#777'>Hits come from this dataset's sample; the "
                             "bill is for your whole area. Read the pattern, not the decimals. "
                             + PRICE_NOTE + "</span>")
                scale = (f"{area.value:,.0f} km², "
                         f"{next(n for n, v in updates.options if v == updates.value)}")
                S["strategy_scale"] = scale
                _, pngs, _ = _run(plot_strategies, rows, k, scale)
                chart.value = pngs[0]
                reading.value = (f"<div style='{_CARD}'>"
                                 + "".join(f"<p style='margin:6px 0'>{x}</p>" for x in lines)
                                 + "</div>")
                for redraw in list(ANSWER_REFRESH.values()):
                    redraw()
            except Exception as exc:
                status.value = _error(exc)
        controls = [area, updates, price_hr, price_sr] + ([screen, share] if has_hr else [])
        for c in controls:
            c.observe(draw, names="value")
        ui = W.VBox([W.HBox([area, area_note]), updates, price_hr, price_sr,
                     *([W.HTML("<br><b>A mixed strategy: screen, then buy</b>"), screen, share]
                       if has_hr else []), status, chart, reading])
        ui.controls = {c.description: c for c in controls}
        ui.chart, ui.reading = chart, reading
        UI["cost"] = ui
        draw()
        return [ui], draw
    _gated("cost", build)


# ---- going further: the SR service -----------------------------------------------------------

OWN_PLACE = "My own place"
SR_CLIENT = "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"


def _secret(name):
    """An API setting from the environment or from Colab's Secrets panel ('' if not there)."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def sr_connect(ready_made, default_lat, default_lon):
    """Type the API key and the service address from your team card (or keep them in Colab's
    Secrets as SR_API_KEY and SR_API_URL), press Connect, then choose an area. Nothing here
    runs before you press the button, so Run all never waits for a key."""
    key = W.Password(value=os.environ.get("SR_API_KEY", ""), description="API key (srk_...)",
                     layout=_WIDE, style=_LABEL)
    url = W.Text(value=os.environ.get("SR_API_URL", ""), description="Service address",
                 layout=_WIDE, style=_LABEL, placeholder="https://...")
    go = W.Button(description="Connect", icon="plug", button_style="primary",
                  layout=W.Layout(width="220px"))
    status = W.HTML()
    panel = W.VBox([])

    def on_go(_):
        go.disabled = True
        try:
            try:
                importlib.import_module("sr_client")
            except ImportError:
                status.value = _note("Getting the SR client (about a minute, once) ...")
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", SR_CLIENT],
                               check=True)
                importlib.invalidate_caches()
            from sr_client import SRClient
            sr = SRClient(key.value.strip() or _secret("SR_API_KEY"),
                          url=url.value.strip() or _secret("SR_API_URL"))
            left = sr.me().get("budget")
            status.value = _note(f"Connected. Budget of this key: {left}", "#1a7f37")
            S["sr"] = sr
            panel.children = (sr_panel(sr, ready_made, default_lat, default_lon, show=False),)
        except Exception as exc:
            status.value = _note(f"Could not connect: {_html.escape(str(exc))}. Check the key "
                                 "and the address on your team card.", "#b00020")
        finally:
            go.disabled = False
    go.on_click(on_go)
    ui = W.VBox([key, url, go, status, panel])
    ui.key, ui.url, ui.go, ui.status, ui.panel = key, url, go, status, panel
    UI["sr_connect"] = ui
    display(ui)


def sr_panel(sr, ready_made, default_lat, default_lon, show=True):
    """Pick a ready-made area (or your own point), send it with one button and look at the
    result with sliders. The button waits until the job is done (3–6 minutes for a new area)."""
    from sr_client import SRError
    from sr_client.client import FINISHED
    place = W.Dropdown(options=[*ready_made, OWN_PLACE], description="Area", layout=_WIDE,
                       style=_LABEL)
    lat = W.FloatText(value=default_lat, description="Latitude", style=_LABEL)
    lon = W.FloatText(value=default_lon, description="Longitude", style=_LABEL)
    km = W.BoundedFloatText(value=2, min=0.5, max=5, step=0.5, description="Size (km)",
                            style=_LABEL)
    day = W.Text(value="2026-03-15", description="Date (YYYY-MM-DD)", style=_LABEL)
    own = W.VBox([W.HTML(_note("A point (in Google Maps: right-click, then click the "
                               "coordinates to copy them), a size and a date, ideally in the "
                               "dry season.")), lat, lon, km, day])
    own.layout.display = "none"
    stages = W.ToggleButtons(options=[("stage 1 and 2", "12"), ("stage 1 only (faster)", "1")],
                             value="12", description="Make", style=_buttons("170px"))
    send = W.Button(description="Send and wait", button_style="primary", icon="rocket",
                    layout=W.Layout(width="220px"))
    status = W.HTML()
    viewer = W.VBox([])

    def toggle(_=None):
        own.layout.display = "" if place.value == OWN_PLACE else "none"
    place.observe(toggle, names="value")

    def on_send(_):
        send.disabled = True
        viewer.children = ()
        try:
            if place.value == OWN_PLACE:
                area, d, options = area_around(lat.value, lon.value, km=km.value), \
                    day.value.strip(), {}
            else:
                p = ready_made[place.value]
                area, d, options = p["area"], p["date"], p.get("options", {})
            job = sr.submit(area, d, stages=list(stages.value), **options)
            t0 = time.monotonic()
            while job.status not in FINISHED:
                status.value = _pre(job._progress_line(time.monotonic() - t0))
                time.sleep(5)
                job.refresh()
            lines = [f"job {job.id[:8]}: {job.status} after "
                     f"{(time.monotonic() - t0) / 60:.1f} min"]
            lines += [f"warning [{w['code']}]: {w['message']}" for w in job.warnings]
            status.value = _pre("\n".join(lines))
            if job.status not in ("done", "partial"):
                status.value += _note(f"The job did not finish: {job.info.get('error')}",
                                      "#b00020")
                return
            S["job"] = job
            viewer.children = (_job_viewer(job),)
        except SRError as exc:
            status.value = _note(f"The service said: {_html.escape(exc.message)}", "#b00020")
        except Exception as exc:
            status.value = _error(exc)
        finally:
            send.disabled = False
    send.on_click(on_send)
    ui = W.VBox([place, own, stages, send, status, viewer])
    ui.place, ui.send, ui.status, ui.viewer, ui.own_fields = place, send, status, viewer, (
        lat, lon, km, day)
    UI["sr"] = ui
    if show:
        display(ui)
    return ui


def _rgb(da):
    a = np.moveaxis(np.nan_to_num(da.isel(band=[0, 1, 2]).values.astype("float32")), 0, -1)
    return a / max(float(a.max()), 1e-6) * 255


def _job_viewer(job):
    """Sentinel-2 and the SR stages side by side, with zoom and position sliders."""
    names = [("Sentinel-2 (10 m)", "lr_reference.tif"), ("SR stage 1", "stage1_sr.tif"),
             ("SR stage 2", "stage2_sr.tif")]
    files = job.results()["files"]
    layers = [(t, _rgb(job.open(f))) for t, f in names if f in files]
    h, w = layers[-1][1].shape[:2]
    px = 2.5
    half = int(min(h, w) * px / 2)
    size = W.ToggleButtons(options=[("250 m", 100), ("500 m", 200), ("1 km", 400),
                                    ("whole area", max(h, w))], value=min(400, max(h, w)),
                           description="Zoom", style=_buttons("90px"))
    east = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                       description="← west | east → (m)", layout=_WIDE, style=_LABEL)
    north = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                        description="← south | north → (m)", layout=_WIDE, style=_LABEL)
    image = W.Image(format="png", layout=W.Layout(max_width="100%"))

    def draw(_=None):
        s = min(size.value, h, w)
        cy = int(np.clip(h / 2 - north.value / px, s / 2, h - s / 2))
        cx = int(np.clip(w / 2 + east.value / px, s / 2, w - s / 2))
        fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 5.3), squeeze=False)
        for ax, (t, a) in zip(axes[0], layers):
            lh, lw = a.shape[:2]
            f = lh / h      # the 10 m image has fewer pixels
            crop = a[int((cy - s / 2) * f):int((cy + s / 2) * f),
                     int((cx - s / 2) * f):int((cx + s / 2) * f)]
            ax.imshow(autostretch(crop), interpolation="nearest")
            ax.set_title(t), ax.set_xticks([]), ax.set_yticks([])
        plt.tight_layout()
        image.value = _png(fig)
    for c in (size, east, north):
        c.observe(draw, names="value")
    draw()
    box = W.VBox([size, east, north, image])
    box.size, box.east, box.north, box.image = size, east, north, image
    return box




# ---- part 6: your recommendation -------------------------------------------------------------

USES = ["to decide where to look first", "as evidence to act on", "not at all"]
SAFEGUARDS = [
    "A person checks every flagged place before anyone acts",
    "Never the only evidence for enforcement, eviction or eligibility",
    "Tested first on places our staff know well",
    "Maps are labelled as AI-enhanced, with the image date",
    "People and local governments can contest a flag",
]


def card_text():
    task = S["task"]

    def said(key):
        v = ANSWERS.get(key)
        return v if v not in ("", None, []) else "(not answered)"
    lines = ["=" * 78, f"TASK      {task.meta['title']}",
             f"BRIEF     {ANSWERS.get('brief', '(none)')}"]
    for k, v in scenario_dict().items():
        lines.append(f"{k.replace('_', ' ').upper():<22}{v}")
    lines += ["-" * 78, "SETTINGS  " + ", ".join(f"{k} {v}" for k, v in S["settings"].items()),
              "", S["res"][[c for c in ("hits", "range over models", "hit rate", "missed places",
                                        "cost") if S["res"][c].astype(str).str.len().max()]]
              .to_string(), ""]
    need = {n: v for n, v in (S.get("visits_needed") or {}).items() if v > S["budget"]}
    if need:
        lines.append("SAME RESULT, MORE VISITS: " + ", ".join(
            f"{v:,} visits with {n}" for n, v in sorted(need.items(), key=lambda x: x[1])))
    if S.get("bets"):
        lines.append("OUR BETS: " + ", ".join(
            f"{n} {b} (result {S['res'].loc[n, 'hits']:g})" for n, b in S["bets"].items()))
    if S.get("strategy"):
        lines.append(f"MONEY ({S['strategy_scale']}):")
        lines += [f"  - {x}" for x in S["strategy"]]
    lines.append("-" * 78)
    lines.append(f"{'recommendation:':<28}{said('VERDICT')}")
    lines.append(f"{'we would use the map:':<28}{said('USE')}")
    guards = ANSWERS.get("SAFEGUARDS") or []
    lines.append(f"{'safeguards:':<28}" + ("(none ticked)" if not guards else ""))
    lines += [f"  - {g}" for g in guards]
    for key, text in (("CHANGE_MIND", "what would change our mind"), ("RESCUE", "where SR helped"),
                      ("FAILURE", "a failure")):
        lines.append(f"{text + ':':<28}{said(key)}")
    lines.append("=" * 78)
    return "\n".join(lines)


def slide_card(fname="my_slide_chart.png"):
    """Part 6: your recommendation, then a card with everything the slide needs; it follows the
    decision settings and your answers. The button downloads the chart."""
    if not _need_decision():
        return

    def build():
        form = ask(("VERDICT", "Your recommendation to the decision-maker",
                    ["deploy", "pilot", "don't use"]),
                   ("USE", "How would you use the map?", USES),
                   ("SAFEGUARDS", "Safeguards you insist on (tick all that apply)",
                    "checks", SAFEGUARDS),
                   ("CHANGE_MIND", "What would change your mind?", "text",
                    "a test on 50 places in our own city, checked in the field"), show=False)
        card = W.HTML()
        chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
        button = W.Button(description=f"Download the chart ({fname})", icon="download",
                          layout=W.Layout(width="320px"))
        note = W.HTML()

        def update(_=None):
            card.value = _pre(card_text())
            guess = S.get("bets")
            fig = plot_decision(S["res"], S["budget"], scenario_dict()["decision"], guess)
            fig.savefig(fname, dpi=200, bbox_inches="tight")
            chart.value = _png(fig)

        def download(_):
            update()
            try:
                from google.colab import files
                files.download(fname)
                note.value = _note("Downloading ...", "#1a7f37")
            except Exception:  # not in Colab
                note.value = _note(f"Saved as {fname} next to this notebook.", "#1a7f37")
        button.on_click(download)
        ANSWER_REFRESH["card"] = update
        update()
        ui = W.VBox([form, W.HTML("<b>Your card</b> (copy it onto your team slide)"), card,
                     chart, button, note])
        ui.card, ui.chart, ui.button, ui.form = card, chart, button, form
        UI["card"], UI["verdict"] = ui, form
        return [ui], update
    _gated("card", build)


# ---- the ready-made areas and the image levels of the "your scenario" notebook ---------------
# Plain data, used three ways: pasted into the notebook by build.py, imported by make_areas.py
# (which runs the areas on the SR service and packs the pictures) and read by the tests.
import numpy as np

# key, name on screen, tau of stage 2 (None: no stage 2). nfe = tau, so cost scales with tau.
LEVELS = [("s2", "Sentinel-2 (10 m, free)", None),
          ("stage1", "SR stage 1 (careful)", None),
          ("t100", "SR stage 2, light", 100),
          ("t250", "SR stage 2, standard", 250),
          ("t500", "SR stage 2, strong", 500)]

# slug: name, what you find there, centre, size (km), date (dry season where we could), and the
# notebook with measured results for this very place, if there is one.
AREAS = {
    # Philippines
    "tondo": dict(name="Tondo and Baseco, Manila", theme="Dense city, port, informal settlements",
                  lat=14.5960, lon=120.9620, km=3, date="2026-03-15"),
    "payatas": dict(name="Payatas, Quezon City", theme="Settlements at the city edge",
                    lat=14.7130, lon=121.1000, km=3, date="2026-03-15"),
    "manila_south": dict(name="Manila South (labelled test area)",
                         theme="Dense city; informal settlements outlined by experts",
                         lat=14.5560, lon=121.0136, km=2, date="2025-06-01", measured="ifs"),
    "sjdm": dict(name="San Jose del Monte, Bulacan", theme="City edge: new housing estates",
                 lat=14.8139, lon=121.0453, km=3, date="2026-03-15"),
    "marikina": dict(name="Marikina River, Marikina City", theme="River and flood plain in a city",
                     lat=14.6350, lon=121.0900, km=3, date="2026-03-15"),
    "rodriguez": dict(name="Rodriguez and San Mateo, Rizal", theme="Quarries at the forest edge",
                      lat=14.7450, lon=121.1500, km=3, date="2026-03-15"),
    "hagonoy": dict(name="Hagonoy, Bulacan", theme="Fishponds and coast (expect a water warning)",
                    lat=14.8340, lon=120.7330, km=3, date="2026-03-15"),
    "munoz": dict(name="Science City of Muñoz, Nueva Ecija", theme="Small rice farms, farm roads",
                  lat=15.7150, lon=120.9050, km=3, date="2026-03-15"),
    "nueva_ecija": dict(name="Nueva Ecija (as in practical demonstration 2)",
                        theme="Rice farms and a town", lat=15.5784, lon=120.9726, km=2,
                        date="2026-03-15"),
    "new_clark": dict(name="New Clark City, Tarlac", theme="Large construction sites, new roads",
                      lat=15.3450, lon=120.5300, km=3, date="2026-03-15"),
    "tacloban": dict(name="Tacloban City, Leyte", theme="Coastal city rebuilt after a typhoon",
                     lat=11.2433, lon=125.0040, km=3, date="2026-04-15"),
    # other countries
    "hanoi": dict(name="Hanoi, Red River banks (labelled area)",
                  theme="City edge: villages, rice fields, building sites",
                  lat=21.0787, lon=105.7822, km=3, date="2025-11-15", measured="lc"),
    "phnom_penh": dict(name="Phnom Penh, southern edge", theme="City growing into wetland",
                       lat=11.5000, lon=104.9200, km=3, date="2026-02-01"),
    "jakarta": dict(name="North Jakarta coast", theme="Dense coastal city, port",
                    lat=-6.1150, lon=106.8000, km=3, date="2026-08-15"),
    "dhaka": dict(name="Dhaka, eastern edge", theme="City growing into flood plain",
                  lat=23.7800, lon=90.4500, km=3, date="2026-01-15"),
}


def stretch_u8(arrays, ref=None):
    """Float RGB arrays (H, W, 3; any range, NaN = no data) to uint8 with ONE contrast stretch
    for all of them, taken from ``ref`` (default: stage 1, else the first). So the levels can
    be compared: none is brightened more than another."""
    ref = ref or ("stage1" if "stage1" in arrays else next(iter(arrays)))
    r = np.nan_to_num(np.asarray(arrays[ref], dtype="float32"))
    lo, hi = np.percentile(r[r.sum(-1) > 0] if (r.sum(-1) > 0).any() else r.reshape(-1, 3),
                           [0.5, 99.5], axis=0)
    out = {}
    for k, a in arrays.items():
        a = np.nan_to_num(np.asarray(a, dtype="float32"))
        out[k] = (np.clip((a - lo) / np.maximum(hi - lo, 1e-6), 0, 1) * 255).round().astype("uint8")
    return out


# ---- "your scenario": the team writes the brief, then judges whether SR fits -------------------
# Pasted after hk.py, hk_widgets.py and areas.py by build.py (scenario notebook only).
# Parts: 1 scenario canvas, 2 area and dates, 3 money, 4 look at the levels, 5 cheat sheet and
# test plan, 6 slide card. Nothing waits for input; the service is needed only for an own area
# (or if the ready-made pictures did not download).

LEVEL_NAME = {k: n for k, n, _ in LEVELS}
LEVEL_TAU = {k: t for k, _, t in LEVELS if t}
PKG = {}          # ready-made areas whose pictures came with the download: slug -> facts
OWN = "own"
_COLAB = "https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/hackathon/"
MEASURED = {
    "ifs": ("Informal settlements, Metro Manila",
            _COLAB + "informal_settlements/hackathon_informal_settlements.ipynb"),
    "lc": ("Land cover around Hanoi", _COLAB + "land_cover/hackathon_land_cover.ipynb"),
    "ai4": ("Field boundaries of smallholder farms",
            _COLAB + "field_boundaries/hackathon_field_boundaries.ipynb"),
}
# computing cost of the SR service per km² of a large area (240 m between tiles; stage 2 costs
# tau steps per tile). Sentinel-2 itself is free.
SR_USD_KM2 = {"stage1": 0.0002, "t100": 0.005, "t250": 0.013, "t500": 0.026}


def get_areas(base_url, name="scenario_areas"):
    """Download the pictures of the ready-made areas once. Without them the notebook still
    works: the areas then come from the SR service (part 2)."""
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    marker = DATA_ROOT / f".{name}.done"
    if not marker.exists():
        tmp = DATA_ROOT / f"{name}.zip"
        print(f"Downloading {name}.zip ...")
        try:
            _fetch(f"{str(base_url).rstrip('/')}/{name}.zip", tmp)
            with zipfile.ZipFile(tmp) as z:
                z.extractall(DATA_ROOT)
            tmp.unlink()
            marker.touch()
        except Exception as e:
            print(f"  (the ready-made pictures did not download: {e}. The notebook works "
                  "without them: connect to the SR service in part 2.)")
    PKG.clear()
    meta = DATA_ROOT / name / "meta.json"
    if meta.exists():
        PKG.update({k: v for k, v in json.loads(meta.read_text())["areas"].items() if k in AREAS})
    print(f"Ready-made areas with pictures: {len(PKG)} of {len(AREAS)}.")


def _pkg_layers(slug, name="scenario_areas"):
    out = {}
    for k, _, _ in LEVELS:
        for ext in ("jpg", "png"):
            p = DATA_ROOT / name / slug / f"{k}.{ext}"
            if p.exists():
                out[k] = np.asarray(Image.open(p).convert("RGB"))
                break
    return out


def _card(html_text, colour="#007DB7"):
    return f"<div style='{_CARD.replace('#007DB7', colour)}'>{html_text}</div>"


def _said(key):
    v = ANSWERS.get(key)
    return v if v not in ("", None, []) else "(not answered)"


# ---- part 1: the scenario canvas ---------------------------------------------------------------

CANVAS = [
    ("team", "Team name", "Team A"),
    ("decision_maker", "Who decides?", "City housing office, settlement upgrading programme"),
    ("decision", "What do they decide, and how often?",
     "Which blocks get a field validation, every quarter"),
    ("today", "How do they decide today?", "From a list that is six years old, and complaints"),
    ("object", "What must the image show?", "Dense small roofs with no streets between them"),
    ("metric", "How will you measure success?", "Of 20 visits, how many find a settlement"),
    ("minimum", "The minimum you need, and why",
     "10 of 20: below that, the teams stop trusting the map"),
    ("below", "What does an error cost, and who pays?",
     "A wasted team-day; families left off the list for a year"),
]
UPDATES = [("once a year", 1), ("each season", 2), ("every quarter", 4), ("every month", 12)]

# Our own scenarios, kept in the back pocket: a team that is stuck can start from one.
STARTERS = [
    dict(title="Housing office: which blocks do the validation teams visit first?",
         task="settlements", area="tondo", size_m=5, area_km2=620, updates=4, fields=dict(
             decision_maker="Regional office for housing and urban development, settlement "
                            "upgrading programme",
             decision="Which 20 city blocks get a field validation, every quarter",
             today="From a list of informal settlements that is years old",
             object="Dense small roofs with no streets between them (blocks of 100 m)",
             metric="Of 20 visits, how many find an informal settlement",
             minimum="10 of 20: below that, the teams go back to the old list",
             below="A wasted team-day; families left out of the shelter plan for a year")),
    dict(title="Space agency: a settlement watch list for local governments?",
         task="settlements", area="payatas", size_m=5, area_km2=300000, updates=4, fields=dict(
             decision_maker="National space agency, space applications team",
             decision="Whether to send every city a list of likely settlement blocks, each quarter",
             today="No regular list; cities ask one by one",
             object="Blocks where informal housing is likely",
             metric="Of the top 50 blocks on a city's list, how many are right",
             minimum="35 of 50: each wrong flag on a formal neighbourhood costs trust",
             below="Local staff stop reading the list")),
    dict(title="Environment department: which building sites get a dust inspection?",
         task="landcover", area="new_clark", size_m=20, area_km2=3360, updates=12, fields=dict(
             decision_maker="City environment department, air-quality enforcement unit",
             decision="Which 20 places get a dust-control inspection, every month",
             today="Inspectors go where complaints come from",
             object="Open, bare ground: building sites, cleared land, sand yards",
             metric="Of 20 inspections, how many find open bare ground",
             minimum="14 of 20: complaints alone already give about 10",
             below="A wasted inspector-day; a dust source uncontrolled for a month")),
    dict(title="Zoning office: are houses going up on protected farmland?",
         task="buildings", area="sjdm", size_m=8, area_km2=3360, updates=4, fields=dict(
             decision_maker="Provincial land-use and zoning office, compliance team",
             decision="Which 15 farmland sites get a check for buildings without a permit, "
                      "every quarter",
             today="No recent map; tips from neighbours",
             object="Single houses (about 8 m) among fields",
             metric="Of 15 site checks, how many find a building on farmland",
             minimum="8 of 15, and no notice is ever sent from the image alone",
             below="A wasted inspector-day; a building stands for another quarter")),
    dict(title="Public works: which parts of the road map are out of date?",
         task="roads", area="munoz", size_m=4, area_km2=5000, updates=1, fields=dict(
             decision_maker="Public works department, planning division",
             decision="Which map cells the road mappers redraw first, once a year",
             today="The inventory misses many farm-to-market roads",
             object="Roads 3 to 6 m wide, many narrower than a Sentinel-2 pixel",
             metric="Of 40 cells redrawn, how many hold a road missing from the inventory",
             minimum="25 of 40",
             below="Half a mapper-day; a road stays off the maintenance list for a year")),
    dict(title="Parks office: where is the tree cover we must protect?",
         task="landcover", area="marikina", size_m=30, area_km2=640, updates=1, fields=dict(
             decision_maker="Parks and green space office, urban greening team",
             decision="Which 20 places get a tree survey and a protection order, once a year",
             today="An old vegetation map",
             object="Groups of trees of 30 m and more",
             metric="Of 20 surveys, how many find dense tree cover",
             minimum="18 of 20: free Sentinel-2 may already reach that",
             below="A wasted surveyor-day; a grove stays unprotected")),
    dict(title="Agriculture office: which villages get the farmer-registry teams?",
         task="fields", area="munoz", size_m=5, area_km2=5000, updates=2, fields=dict(
             decision_maker="Department of agriculture regional office, smallholder programme",
             decision="Which 30 areas the extension workers visit to enrol farmers, each season",
             today="No map of the fields; the smallest farms are missing from the registry",
             object="Field boundaries: bunds and paths 1 to 3 m wide between small fields",
             metric="Of 30 visits, how many reach an area with many small farms",
             minimum="20 of 30",
             below="A wasted worker-day; about 30 smallholders not enrolled this season")),
    dict(title="Statistics office: where does the farm census need a second enumerator?",
         task="fields", area="nueva_ecija", size_m=5, area_km2=5000, updates=1, fields=dict(
             decision_maker="National statistics office, agricultural statistics division",
             decision="Which 20 square kilometres get a second enumerator, once per census",
             today="A parcel count that is ten years old",
             object="Parcels, to count them per square kilometre",
             metric="Of 20 squares chosen, how many really have very many parcels",
             minimum="15 of 20",
             below="An enumerator sent where one was enough; a square counted late")),
    dict(title="Disaster office: how far did the flood reach?",
         task="flood", area="marikina", size_m=20, area_km2=640, updates=12, fields=dict(
             decision_maker="City disaster risk reduction and management office",
             decision="Which barangays get relief and damage assessment teams, after each flood",
             today="Reports by phone from barangay captains",
             object="The edge of the water, to the nearest street (about 20 m)",
             metric="Share of flooded barangays that the map shows as flooded",
             minimum="9 of 10: a missed barangay waits a day longer for help",
             below="Teams sent to dry streets; flooded families reached late")),
    dict(title="Mining regulator: which quarries dig outside their permit?",
         task="landcover", area="rodriguez", size_m=15, area_km2=1200, updates=4, fields=dict(
             decision_maker="Provincial mining regulatory board",
             decision="Which 20 quarry sites get an inspection, every quarter",
             today="Inspections by rotation, each site about every two years",
             object="Freshly dug ground next to a permit boundary (changes of 15 m and more)",
             metric="Of 20 inspections, how many find digging outside the permit area",
             minimum="8 of 20: rotation finds about 3",
             below="A wasted inspector-day; illegal digging continues for a quarter")),
    dict(title="Fisheries office: which fishponds are in use?",
         task="flood", area="hagonoy", size_m=10, area_km2=900, updates=2, fields=dict(
             decision_maker="Provincial fisheries office",
             decision="Which pond owners get a visit about lease renewal, each season",
             today="Lease records on paper; no map of which ponds hold water",
             object="Pond dikes (2 to 5 m wide) and whether a pond holds water",
             metric="Of 30 visits, how many ponds are in the state the map showed",
             minimum="24 of 30",
             below="A wasted visit by boat; a lapsed lease goes unnoticed")),
]


def canvas():
    """Part 1: the team's own scenario. Every field saves itself. 'Stuck?' offers our own
    scenarios as a starting point; they fill the fields, which stay editable."""
    form = ask(*[(k, t, "text", ex) for k, t, ex in CANVAS], show=False)
    size = W.BoundedFloatText(value=5, min=0.3, max=500, step=0.5, style=_LABEL,
                              description="Smallest thing you must see (m)")
    area = W.BoundedFloatText(value=600, min=1, max=1e8, step=10, style=_LABEL,
                              description="Area you must watch (km²)")
    updates = W.ToggleButtons(options=UPDATES, value=4, description="A fresh map",
                              style=_buttons("110px"))
    hint = W.HTML()

    def on_num(_=None):
        ANSWERS.update(size_m=size.value, area_km2=area.value, updates=updates.value)
        hint.value = _note(_size_rule(size.value), "#555")
        for redraw in list(ANSWER_REFRESH.values()):
            redraw()
    for c in (size, area, updates):
        c.observe(on_num, names="value")

    pick = W.Dropdown(options=[("(choose one)", -1)] + [(s["title"], i)
                                                        for i, s in enumerate(STARTERS)],
                      value=-1, layout=W.Layout(width="720px"))
    fill = W.Button(description="Fill the canvas with it", icon="clipboard",
                    layout=W.Layout(width="260px"))
    told = W.HTML()

    def on_fill(_):
        if pick.value < 0:
            return
        s = STARTERS[pick.value]
        for k, f in form.fields.items():
            if k != "team":
                f.value = s["fields"].get(k, "")
        size.value, area.value, updates.value = s["size_m"], s["area_km2"], s["updates"]
        ANSWERS["starter"] = s["title"]
        for name, key in (("area", "place"), ("cheat", "task")):
            if name in UI and hasattr(UI[name], key):
                try:
                    getattr(UI[name], key).value = s["area" if name == "area" else "task"]
                except Exception:
                    pass
        told.value = _note("Filled. Now change every field until it describes a decision in "
                           "<b>your</b> agency. Part 2 suggests an area for it.", "#1a7f37")
    fill.on_click(on_fill)
    stuck = W.Accordion(children=[W.VBox([
        W.HTML(_note("Scenarios we prepared. The kind of decision is real; the offices and "
                     "the numbers are invented.")), pick, fill, told])])
    stuck.set_title(0, "Stuck? Start from one of ours")
    stuck.selected_index = None
    ui = W.VBox([form, W.HTML("<b>Three numbers the later parts use</b>"), size, hint, area,
                 updates, stuck])
    ui.fields, ui.size, ui.area, ui.updates = form.fields, size, area, updates
    ui.pick, ui.fill = pick, fill
    UI["canvas"] = ui
    on_num()
    display(ui)


def _size_rule(m):
    if m < 2.5:
        return (f"{m:g} m is smaller than one SR pixel (2.5 m). SR cannot show it: what it draws "
                "at that size is invented. Plan for real high-resolution images.")
    if m < 10:
        return (f"{m:g} m lies between the SR pixel (2.5 m) and the Sentinel-2 pixel (10 m). "
                "This is where SR can help, and where it can invent. Test before you trust.")
    if m < 20:
        return (f"{m:g} m is one to two Sentinel-2 pixels. SR may sharpen it; try plain "
                "Sentinel-2 first.")
    return f"{m:g} m is several Sentinel-2 pixels. Free Sentinel-2 probably shows it already."


def canvas_dict():
    return {k: (ANSWERS.get(k) or "").strip() or "(not answered)" for k, _, _ in CANVAS}


# ---- part 2: area and dates --------------------------------------------------------------------

def _scene_html(info):
    """Scenes found, tiles, cost and warnings, from the service's estimate or the package."""
    scenes = info.get("scenes") or []
    dates = sorted({s["date"] for s in scenes})
    rows = [f"<b>Sentinel-2 images found:</b> {len(scenes)} scenes on {len(dates)} days"
            + (f", {dates[0]} to {dates[-1]}" if dates else "")]
    clouds = [s["cloud_cover"] for s in scenes if s.get("cloud_cover") is not None]
    if clouds:
        rows.append(f"<b>Cloud cover of those scenes:</b> {min(clouds):.0f} % to "
                    f"{max(clouds):.0f} % (median {float(np.median(clouds)):.0f} %)")
    if len(dates) < 4:
        rows.append("<span style='color:#b00020'><b>Fewer than 4 clear days.</b> Stage 1 "
                    "blends several images; with so few it has little to work with. Try "
                    "another date, ideally in the dry season.</span>")
    if info.get("tiles"):
        rows.append(f"<b>Work for the service:</b> {info['tiles']} tiles")
    usd = info.get("usd") or {}
    if usd:
        rows.append("<b>Computing cost of this area:</b> " + ", ".join(
            f"{LEVEL_NAME[k].replace('SR ', '')} USD {v:.2f}" for k, v in usd.items()))
    codes = [w if isinstance(w, str) else w.get("code", "?")
             for w in info.get("warnings") or []]
    if codes:
        rows.append("<b>Warnings:</b> " + ", ".join(codes)
                    + ("  (water: SR erases detail in shallow water; use Sentinel-2 there)"
                       if "water" in codes else ""))
    for p in info.get("problems") or []:
        rows.append(f"<span style='color:#b00020'><b>Problem:</b> {_html.escape(str(p))}</span>")
    return "<br>".join(rows)


def _connect_box(on_connected):
    key = W.Password(value=os.environ.get("SR_API_KEY", ""), description="API key (srk_...)",
                     layout=_WIDE, style=_LABEL)
    url = W.Text(value=os.environ.get("SR_API_URL", ""), description="Service address",
                 layout=_WIDE, style=_LABEL, placeholder="https://...")
    go = W.Button(description="Connect", icon="plug", button_style="primary",
                  layout=W.Layout(width="220px"))
    status = W.HTML()

    def on_go(_):
        go.disabled = True
        try:
            try:
                importlib.import_module("sr_client")
            except ImportError:
                status.value = _note("Getting the SR client (about a minute, once) ...")
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", SR_CLIENT],
                               check=True)
                importlib.invalidate_caches()
            from sr_client import SRClient
            sr = SRClient(key.value.strip() or _secret("SR_API_KEY"),
                          url=url.value.strip() or _secret("SR_API_URL"))
            left = sr.me().get("budget")
            S["sr"] = sr
            status.value = _note(f"Connected. Budget of this key: {left}", "#1a7f37")
            on_connected()
        except Exception as exc:
            status.value = _note(f"Could not connect: {_html.escape(str(exc))}. Check the key "
                                 "and the address on your team card.", "#b00020")
        finally:
            go.disabled = False
    go.on_click(on_go)
    box = W.VBox([W.HTML(_note("The API key and the service address are on your team card.")),
                  key, url, go, status])
    box.key, box.url, box.go, box.status = key, url, go, status
    return box


def area_panel():
    """Part 2: choose a ready-made area (pictures come with the notebook) or a place of your
    own (the SR service makes the pictures; send before lunch). Shows which Sentinel-2 images
    exist and what the computing costs."""
    themed = [(f"{a['theme']}  ·  {a['name']}" + ("  ★" if a.get("measured") else ""), slug)
              for slug, a in AREAS.items()]
    place = W.Dropdown(options=themed + [("A place of our own (needs the SR service)", OWN)],
                       value=themed[0][1], description="Area", layout=W.Layout(width="760px"),
                       style={"description_width": "60px"})
    info = W.HTML()
    lat = W.FloatText(value=14.5960, description="Latitude", style=_LABEL)
    lon = W.FloatText(value=120.9620, description="Longitude", style=_LABEL)
    km = W.BoundedFloatText(value=2, min=0.5, max=2, step=0.5, description="Size (km, up to 2)",
                            style=_LABEL)
    day = W.Text(value="2026-03-15", description="Date (YYYY-MM-DD)", style=_LABEL)
    own = W.VBox([W.HTML(_note(
        "A point (in Google Maps: right-click, then click the coordinates to copy them), a "
        "size and a date. Clear skies matter: pick the dry season. A new area takes about "
        "5 to 10 minutes for all three levels, so <b>send it before lunch</b>.")),
        lat, lon, km, day])
    check = W.Button(description="Check Sentinel-2 images (free)", icon="search",
                     layout=W.Layout(width="300px"))
    send = W.Button(description="Send at three levels", icon="rocket", button_style="primary",
                    layout=W.Layout(width="300px"))
    status = W.HTML()

    def chosen():
        if place.value == OWN:
            return dict(slug=OWN, name=f"Our own area ({lat.value:.4f}, {lon.value:.4f})",
                        lat=lat.value, lon=lon.value, km=km.value, date=day.value.strip())
        return dict(slug=place.value, **AREAS[place.value])

    def refresh(_=None):
        a = chosen()
        S["area"] = a
        S.pop("layers", None)
        ANSWERS["area"] = f"{a['name']}, {a['km']:g} km × {a['km']:g} km, around {a['date']}"
        ready = a["slug"] in PKG
        own.layout.display = "" if a["slug"] == OWN else "none"
        if a["slug"] == OWN:
            info.value = ""
        else:
            lines = [f"<b>{a['name']}</b>: {a['theme'].lower()}. {a['km']} km × {a['km']} km, "
                     f"images around {a['date']}."]
            if a.get("measured"):
                t, link = MEASURED[a["measured"]]
                lines.append(f"★ Experts labelled this place: measured results are in the "
                             f"notebook <a href='{link}' target='_blank'>{t}</a>.")
            if ready:
                S["scene_info"] = PKG[a["slug"]]
                lines += [_scene_html(PKG[a["slug"]]),
                          "<span style='color:#1a7f37'><b>The pictures are ready</b>: go on "
                          "to part 3. No key needed.</span>"]
            else:
                S.pop("scene_info", None)
                lines.append("<span style='color:#8a6d00'>The pictures of this area are not in "
                             "the download. Connect to the SR service below and press "
                             "<b>Send</b>: areas we ran before the workshop come back "
                             "at once.</span>")
            info.value = _card("<br>".join(lines))
        service.layout.display = "none" if ready else ""
        for redraw in list(ANSWER_REFRESH.values()):
            redraw()

    def need_sr():
        if "sr" not in S:
            status.value = _note("Connect first: type the key and the address, then press "
                                 "Connect.", "#b00020")
            return None
        return S["sr"]

    def on_check(_):
        sr = need_sr()
        if not sr:
            return
        check.disabled = True
        try:
            a = chosen()
            box = area_around(a["lat"], a["lon"], km=a["km"])
            out = {"usd": {}}
            for k, tau in LEVEL_TAU.items():
                est = sr.estimate(box, a["date"], tau=tau)
                out["usd"][k] = est.get("estimated_usd", 0)
                out.update({f: est.get(f) for f in ("scenes", "tiles", "problems")})
                out["warnings"] = est.get("warnings_likely", [])
            S["scene_info"] = out
            status.value = _card(_scene_html(out))
            for redraw in list(ANSWER_REFRESH.values()):
                redraw()
        except Exception as exc:
            status.value = _note(f"The service said: {_html.escape(str(exc))}", "#b00020")
        finally:
            check.disabled = False

    def on_send(_):
        sr = need_sr()
        if not sr:
            return
        send.disabled = True
        try:
            a = chosen()
            box = area_around(a["lat"], a["lon"], km=a["km"])
            jobs = {}
            for k, tau in LEVEL_TAU.items():
                jobs[k] = sr.submit(box, a["date"], tau=tau)
            S["jobs"], S["jobs_area"] = jobs, a
            status.value = _card(
                "<b>Sent.</b> " + ", ".join(f"{LEVEL_NAME[k]}: job {j.id[:8]} ({j.status})"
                                            for k, j in jobs.items())
                + "<br>The service works while you are away. After lunch, press <b>Load the "
                  "pictures</b> in part 4.<br><span style='color:#777'>If this notebook "
                  "restarts over lunch: choose the same area and date and press Send again. "
                  "A request the service has already computed comes back at once and costs "
                  "nothing.</span>", "#1a7f37")
        except Exception as exc:
            status.value = _note(f"The service said: {_html.escape(str(exc))}", "#b00020")
        finally:
            send.disabled = False

    connect = _connect_box(lambda: None)
    service = W.VBox([W.HTML("<b>The SR service</b>"), connect, own, W.HBox([check, send]),
                      status])
    for c in (place, lat, lon, km, day):
        c.observe(refresh, names="value")
    check.on_click(on_check)
    send.on_click(on_send)
    ui = W.VBox([place, W.HTML(_note("★ = a place where experts labelled what is on the "
                                     "ground, so results were measured.", "#777")),
                 info, service])
    ui.place, ui.info, ui.connect, ui.check, ui.send, ui.status = (place, info, connect, check,
                                                                   send, status)
    ui.own_fields = (lat, lon, km, day)
    UI["area"] = ui
    refresh()
    display(ui)


# ---- part 3: money -----------------------------------------------------------------------------

def money_panel():
    """Part 3: the imagery bill for the whole area, per kind of image, and what real images
    would have to save to be worth their price."""
    c = UI.get("canvas")
    price_hr = W.BoundedFloatText(value=10, min=0, max=1000, step=1, style=_LABEL,
                                  description="Real images, USD per km²")
    min_order = W.BoundedFloatText(value=25, min=0, max=10000, step=25, style=_LABEL,
                                   description="Smallest order (km²)")
    level = W.ToggleButtons(options=[(LEVEL_NAME[k].replace("SR ", ""), k) for k in SR_USD_KM2],
                            value="t250", description="SR level", style=_buttons("150px"))
    price_sr = W.BoundedFloatText(value=SR_USD_KM2["t250"], min=0, max=100, step=0.001,
                                  style=_LABEL, description="Super-resolution, USD per km²")
    share = W.IntSlider(value=5, min=1, max=50, continuous_update=False, layout=_WIDE,
                        style=_LABEL, description="Mixed: real images for (%)")
    wasted = W.BoundedFloatText(value=150, min=1, max=1e6, step=10, style=_LABEL,
                                description="One wasted action costs (USD)")
    chart = W.Image(format="png", layout=W.Layout(max_width="100%"))
    reading, status = W.HTML(), W.HTML()

    def on_level(_):
        price_sr.value = SR_USD_KM2[level.value]
    level.observe(on_level, names="value")

    def draw(_=None):
        try:
            status.value = ""
            area = float(ANSWERS.get("area_km2", 600))
            n = int(ANSWERS.get("updates", 4))
            every = next(t for t, v in UPDATES if v == n)
            sr = area * price_sr.value * n
            real = max(area, min_order.value) * price_hr.value * n
            part = max(area * share.value / 100, min_order.value) * price_hr.value * n
            mixed = sr + part
            name = LEVEL_NAME[level.value].replace(" (careful)", "")
            rows = [("Sentinel-2 only", 0.0, COLOURS["s2"]), (name, sr, COLOURS["stage2"]),
                    (f"{name} everywhere,\nreal images for {share.value} %", mixed, "#f2a900"),
                    ("Real images everywhere", real, COLOURS["hr"])]
            fig, ax = plt.subplots(figsize=(8.6, 2.9))
            top = max(real, mixed, sr, 1)
            for i, (label, usd, colour) in enumerate(rows):
                ax.barh(i, max(usd, top * 0.004), color=colour)
                ax.text(max(usd, top * 0.004) + top * 0.01, i,
                        "free" if usd == 0 else f"USD {usd:,.0f}", va="center")
            ax.set_yticks(range(len(rows)))
            ax.set_yticklabels([r[0] for r in rows])
            ax.invert_yaxis()
            ax.set_xlim(0, top * 1.2)
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
            ax.set_xlabel(f"imagery bill per year (USD) for {area:,.0f} km², {every}")
            chart.value = _png(fig)
            gap = real - sr
            visits = gap / wasted.value
            lines = [
                f"<b>Per year:</b> Sentinel-2 free; {name} about <b>USD {sr:,.0f}</b>; real "
                f"images about <b>USD {real:,.0f}</b>"
                + (f" (the smallest order, {min_order.value:,.0f} km², is larger than your "
                   "area)" if min_order.value > area else "") + ".",
                f"<b>What must real images save?</b> They cost USD {gap:,.0f} a year more than "
                f"{name}. At USD {wasted.value:,.0f} per wasted action, that is the price of "
                f"<b>{visits:,.0f} wasted actions a year</b> ({visits / n:,.0f} per update). "
                "Real images are worth it only if they avoid more than that.",
                f"<b>A mixed strategy:</b> {name} everywhere to choose where to look, real "
                f"images for {share.value} % of the area: about <b>USD {mixed:,.0f}</b> a year.",
            ]
            info = S.get("scene_info") or {}
            a = S.get("area") or {}
            if info.get("usd") and a.get("km"):
                lines.append("<span style='color:#777'>Your area in part 2 "
                             f"({a['km'] ** 2:g} km²) costs USD "
                             f"{info['usd'].get(level.value, 0):.2f} at this level. Small jobs "
                             "cost more per km² than large ones: each job starts a computer."
                             "</span>")
            lines.append("<span style='color:#777'>Starting prices: real images about USD 4–19 "
                         "per km² at list price (Sessions 1 and 3), often with a smallest "
                         "order of 25–100 km²; super-resolution is the computing cost of the "
                         "service (Session 4). People, field checks and storage are extra for "
                         "every option. Change the numbers to yours.</span>")
            reading.value = _card("".join(f"<p style='margin:6px 0'>{x}</p>" for x in lines))
            S["money"] = [f"{area:,.0f} km², {every}",
                          f"Sentinel-2 free; {name} USD {sr:,.0f}; real images USD {real:,.0f} "
                          "a year",
                          f"real images must avoid {visits:,.0f} wasted actions a year "
                          f"(USD {wasted.value:,.0f} each) to pay off",
                          f"mixed ({name} + real images for {share.value} %): USD {mixed:,.0f} "
                          "a year"]
            if "scenario_card" in ANSWER_REFRESH:
                ANSWER_REFRESH["scenario_card"]()
        except Exception as exc:
            status.value = _error(exc)
    controls = [price_hr, min_order, level, price_sr, share, wasted]
    for w in controls:
        w.observe(draw, names="value")
    seen = {}

    def on_answer():          # only the numbers of part 1 matter here, not every keystroke
        now = (ANSWERS.get("area_km2"), ANSWERS.get("updates"), id(S.get("scene_info")))
        if seen.get("now") != now:
            seen["now"] = now
            draw()
    ANSWER_REFRESH["money"] = on_answer
    numbers = W.HTML(_note("The area and how often you need a fresh map come from part 1.",
                           "#777"))
    ui = W.VBox([numbers, price_hr, min_order, level, price_sr, share, wasted, status, chart,
                 reading])
    ui.controls = {w.description: w for w in controls}
    ui.chart, ui.reading = chart, reading
    UI["money"] = ui
    draw()
    display(ui)


# ---- part 4: look at the levels ----------------------------------------------------------------

def _job_layers(jobs):
    """Finished jobs (level -> job) to display layers. Sentinel-2 and stage 1 come from the
    first job; stage 2 from each."""
    raw = {}

    def rgb(job, name):
        da = job.open(name)
        return np.moveaxis(da.isel(band=[0, 1, 2]).values.astype("float32"), 0, -1)
    first = next(iter(jobs.values()))
    files = first.results()["files"]
    if "lr_reference.tif" in files:
        raw["s2"] = rgb(first, "lr_reference.tif")
    if "stage1_sr.tif" in files:
        raw["stage1"] = rgb(first, "stage1_sr.tif")
    for k, job in jobs.items():
        if "stage2_sr.tif" in job.results()["files"]:
            raw[k] = rgb(job, "stage2_sr.tif")
    return stretch_u8(raw)


def _compare_viewer(layers, px=2.5):
    """The levels side by side (or two of them, large), with zoom and position sliders."""
    keys = [k for k, _, _ in LEVELS if k in layers]
    ref = max(keys, key=lambda k: layers[k].shape[0])
    h, w = layers[ref].shape[:2]
    half = int(min(h, w) * px / 2)
    sizes = [(t, v) for t, v in (("250 m", 100), ("500 m", 200), ("1 km", 400)) if v < max(h, w)]
    size = W.ToggleButtons(options=sizes + [("whole area", max(h, w))],
                           value=200 if max(h, w) > 200 else max(h, w), description="Zoom",
                           style=_buttons("90px"))
    mode = W.ToggleButtons(options=[("all side by side", "all"), ("two, large", "two")],
                           value="all", description="Show", style=_buttons("130px"))
    names = [(LEVEL_NAME[k], k) for k in keys]
    left = W.Dropdown(options=names, value=keys[0], description="Left", style=_LABEL)
    right = W.Dropdown(options=names, value=keys[-1], description="Right", style=_LABEL)
    pair = W.HBox([left, right])
    pair.layout.display = "none"
    colours = W.ToggleButtons(options=[("same for all", "shared"), ("each its own", "own")],
                              value="shared", description="Contrast", style=_buttons("110px"),
                              tooltips=["one stretch for all images: a fair comparison",
                                        "each image stretched by itself"])
    east = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                       description="← west | east → (m)", layout=_WIDE, style=_LABEL)
    north = W.IntSlider(value=0, min=-half, max=half, step=50, continuous_update=False,
                        description="← south | north → (m)", layout=_WIDE, style=_LABEL)
    image = W.Image(format="png", layout=W.Layout(max_width="100%"))

    def crop(k, cy, cx, s):
        a = layers[k]
        f = a.shape[0] / h           # the 10 m image has fewer pixels
        return a[int((cy - s / 2) * f):int((cy + s / 2) * f),
                 int((cx - s / 2) * f):int((cx + s / 2) * f)]

    def draw(_=None):
        pair.layout.display = "" if mode.value == "two" else "none"
        s = min(size.value, h, w)
        cy = int(np.clip(h / 2 - north.value / px, s / 2, h - s / 2))
        cx = int(np.clip(w / 2 + east.value / px, s / 2, w - s / 2))
        show = keys if mode.value == "all" else list(dict.fromkeys([left.value, right.value]))
        crops = {k: crop(k, cy, cx, s).astype("float32") for k in show}
        if colours.value == "shared":
            base = crop("stage1" if "stage1" in layers else ref, cy, cx, s).astype("float32")
            lo, hi = np.percentile(base.reshape(-1, 3), [2, 98], axis=0)
            crops = {k: np.clip((a - lo) / np.maximum(hi - lo, 1), 0, 1) for k, a in crops.items()}
        else:
            crops = {k: autostretch(a) for k, a in crops.items()}
        wide = 4.2 if mode.value == "all" else 7.2
        fig, axes = plt.subplots(1, len(show), figsize=(wide * len(show), wide + 0.4),
                                 squeeze=False)
        for ax, k in zip(axes[0], show):
            ax.imshow(crops[k], interpolation="nearest")
            ax.set_title(LEVEL_NAME[k]), ax.set_xticks([]), ax.set_yticks([])
        fig.suptitle(f"{s * px:,.0f} m across", y=0.02, fontsize=9, color="#555")
        plt.tight_layout()
        fig.savefig("my_comparison.png", dpi=150, bbox_inches="tight")
        image.value = _png(fig)
    for c in (size, mode, left, right, colours, east, north):
        c.observe(draw, names="value")
    draw()
    box = W.VBox([size, mode, pair, colours, east, north, image])
    box.size, box.mode, box.left, box.right, box.colours = size, mode, left, right, colours
    box.east, box.north, box.image = east, north, image
    return box


LEVEL_CHOICES = ["Sentinel-2", "SR stage 1", "stage 2 light", "stage 2 standard",
                 "stage 2 strong", "none of them: real images"]
_CHOICE_KEY = dict(zip(LEVEL_CHOICES, ["s2", "stage1", "t100", "t250", "t500", "hr"]))


def look_panel():
    """Part 4: load the pictures of your area (ready-made: at once; your own: from the SR
    service), compare the levels and commit to the one you would use."""
    load = W.Button(description="Load the pictures", icon="image", button_style="primary",
                    layout=W.Layout(width="260px"))
    status = W.HTML()
    viewer = W.VBox([])
    save = W.Button(description="Download this view (my_comparison.png)", icon="download",
                    layout=W.Layout(width="360px"))
    note = W.HTML()

    def on_load(_=None):
        load.disabled = True
        try:
            a = S.get("area") or {}
            if a.get("slug") in PKG:
                layers = _pkg_layers(a["slug"])
                status.value = _note(f"{a['name']}: ready-made pictures.", "#1a7f37")
            elif S.get("jobs"):
                from sr_client.client import FINISHED
                jobs, t0 = S["jobs"], time.monotonic()
                while any(j.status not in FINISHED for j in jobs.values()):
                    status.value = _pre("\n".join(
                        f"{LEVEL_NAME[k]:<26}{j._progress_line(time.monotonic() - t0)}"
                        for k, j in jobs.items()))
                    time.sleep(5)
                    for j in jobs.values():
                        if j.status not in FINISHED:
                            j.refresh()
                good = {k: j for k, j in jobs.items() if j.status in ("done", "partial")}
                lines = [f"{LEVEL_NAME[k]}: {j.status}"
                         + "".join(f"; warning [{w['code']}]" for w in j.warnings)
                         + (f"; {j.info.get('error')}" if j.status == "failed" else "")
                         for k, j in jobs.items()]
                status.value = _pre("\n".join(lines))
                if not good:
                    status.value += _note("No level finished. Check the date in part 2 (are "
                                          "there Sentinel-2 images?) and send again.", "#b00020")
                    return
                layers = _job_layers(good)
                info = S.setdefault("scene_info", {})
                info["warnings"] = sorted({w["code"] for j in good.values() for w in j.warnings})
            else:
                status.value = _note("Nothing to load yet. In part 2, choose a ready-made area, "
                                     "or connect to the SR service and press Send.", "#b00020")
                return
            S["layers"] = layers
            viewer.children = (_compare_viewer(layers), save, note)
        except Exception as exc:
            status.value = _error(exc)
        finally:
            load.disabled = False
    load.on_click(on_load)

    def on_save(_):
        try:
            from google.colab import files
            files.download("my_comparison.png")
            note.value = _note("Downloading ...", "#1a7f37")
        except Exception:  # not in Colab
            note.value = _note("Saved as my_comparison.png next to this notebook.", "#1a7f37")
    save.on_click(on_save)
    form = ask(("SEE_S2", "On plain Sentinel-2, can you see what you need?",
                ["yes", "partly", "no"]),
               ("LEVEL", "Which image would you use for this task?", LEVEL_CHOICES),
               ("WHY", "What does it show that you need?", "text",
                "the edges of single roofs; Sentinel-2 shows only a grey patch"),
               ("INVENTED", "Anything that looks invented or wrong? Where?", "text",
                "strong draws streets in the settlement that stage 1 does not show"),
               show=False)
    ui = W.VBox([load, status, viewer,
                 W.HTML("<br><b>Your judgement</b> (decide before you open part 5)"), form])
    ui.load, ui.status, ui.viewer, ui.fields = load, status, viewer, form.fields
    UI["look"] = ui
    display(ui)
    if (S.get("area") or {}).get("slug") in PKG:
        on_load()


# ---- part 5: cheat sheet and test plan ---------------------------------------------------------

# What we measured, by kind of task. best: the levels that measured best (within noise).
CHEAT = {
    "settlements": dict(
        title="Finding where something is concentrated (informal settlements, dense housing)",
        verdict="A fit for choosing where to look. Not for mapping or counting.",
        best=("t250", "t500"), best_text="SR stage 2, standard",
        lines=["Metro Manila, six models that never saw an SR image. Map score (F1): "
               "Sentinel-2 8 %, stage 1 18 %, stage 2 standard 22 %, real images 40 %.",
               "By level: light 15 %, standard 22 %, strong 24 %. Standard and strong are the "
               "same within noise; light is no better than stage 1.",
               "Of 20 field visits, hits: Sentinel-2 6, stage 1 9, stage 2 standard 9, real "
               "images 13. With Sentinel-2 the result depends on the model you happen to have "
               "(1 to 11 hits); with SR far less."],
        watch="False alarms on dense formal housing, markets and warehouses. Real images are "
              "still clearly better.",
        notebook="ifs"),
    "buildings": dict(
        title="Mapping or counting buildings",
        verdict="It helps, but stays far from real images. The best level depends on the "
                "model you reuse.",
        best=("stage1", "t100", "t250"),
        best_text="stage 1 for outlines; stage 2 standard with a model trained on sharp images",
        lines=["Building outlines (SpaceNet 7; a model trained on 4.7 m satellite images): F1 "
               "0.21 on Sentinel-2, 0.27 stage 1, 0.25 stage 2 standard, 0.62 on the real "
               "images. Each stronger level scored lower.",
               "Finding buildings in disaster areas (xBD; a model trained on 0.5 m images): F1 "
               "0.31 on Sentinel-2, 0.48 stage 1, 0.54 standard (light and strong 0.53), 0.79 "
               "on real images.",
               "Houses on farmland around Hanoi, of 15 site checks: Sentinel-2 1, stage 1 5, "
               "stage 2 5, real images 8."],
        watch="On the outline test the models found only about one building pixel in five. "
              "Do not count buildings with SR. Stage 2 draws plausible roofs.",
        notebook="lc"),
    "damage": dict(
        title="Damage to buildings after a disaster",
        verdict="Not a fit for grading damage.",
        best=("hr",), best_text="none of the SR levels: real images",
        lines=["xBD, five disasters. F1 for damaged buildings: 0.21 on Sentinel-2, 0.21 with "
               "stage 2 standard, 0.36 on real images. No gain from SR.",
               "SR does help to find where the buildings are (see \"Mapping or counting "
               "buildings\")."],
        watch="Damage to a single roof is smaller than the SR pixel. A convincing picture is "
              "not evidence.",
        notebook=None),
    "landcover": dict(
        title="Land cover classes (built-up, trees, crops, bare soil, water)",
        verdict="A fit for small or mixed classes. Large uniform classes do not need SR.",
        best=("t100", "t250"), best_text="SR stage 2, light or standard (light is cheaper)",
        lines=["Around Hanoi, seven classes. Map score (mean IoU): Sentinel-2 0.32, stage 1 "
               "0.38, stage 2 standard 0.47, real images 0.57.",
               "By level, on the test tiles: light 0.54, standard 0.53, strong 0.52. The same "
               "within noise.",
               "The decision can differ from the map score. Bare soil, of 20 inspections: "
               "Sentinel-2 12, stage 1 13, stage 2 16, real 17. Trees, of 20: Sentinel-2 18, "
               "real 19: free images are enough."],
        watch="Bright industrial roofs and river sandbanks get flagged as bare soil.",
        notebook="lc"),
    "roads": dict(
        title="Roads and tracks",
        verdict="A fit for finding where roads are. Check narrow roads on a real image.",
        best=("t250",), best_text="SR stage 2, standard",
        lines=["Around Hanoi, of 40 map cells chosen for the most road: Sentinel-2 19, stage 1 "
               "24, stage 2 standard 30, real images 33.",
               "Map score for the road class (IoU): Sentinel-2 0.20, stage 1 0.30, stage 2 "
               "standard 0.34, real images 0.50. Good enough to choose cells, not to draw "
               "the line."],
        watch="Roads under trees stay hidden in any satellite image. Stage 2 can join or "
              "break a road.",
        notebook="lc"),
    "fields": dict(
        title="Field boundaries, small farms, parcels",
        verdict="A fit, with stage 1. Stage 2 adds texture that costs accuracy.",
        best=("stage1",), best_text="SR stage 1",
        lines=["Viet Nam and Cambodia, models trained on each kind of image. Field F1: "
               "Sentinel-2 0.42, stage 1 0.46, light 0.45, standard 0.44, strong 0.44.",
               "Of 30 visits to find small farms: Sentinel-2 13, stage 1 24, stage 2 standard "
               "22. Counting all parcels in 1 km squares, of 20: Sentinel-2 16, stage 1 19: "
               "nearly the same."],
        watch="A hazy area got no visits at all with Sentinel-2. A model needs labels from "
              "your own country.",
        notebook="ai4"),
    "flood": dict(
        title="Water: floods, ponds, shorelines",
        verdict="Not a fit. Free Sentinel-2 does as well.",
        best=("s2",), best_text="plain Sentinel-2",
        lines=["Flood water (FloodPlanet benchmark), map score (mean IoU): Sentinel-2 0.62; SR "
               "at every level 0.61 to 0.62. No gain.",
               "The SR service warns about water: super-resolution erases detail in shallow "
               "water."],
        watch="For floods the date matters more than the detail: you need a clear image on "
              "the right day, and clouds often prevent it.",
        notebook=None),
    "change": dict(
        title="Change between two dates (new buildings, new clearing)",
        verdict="Not measured. Be careful: test first.",
        best=("stage1", "s2"), best_text="stage 1 (our reasoning, not a measurement)",
        lines=["We have not measured change detection.",
               "Stage 2 draws its detail anew on every date, so two stage 2 images can differ "
               "where the ground did not change. Stage 1 invents nothing."],
        watch="Compare stage 1 with stage 1, or Sentinel-2 with Sentinel-2; confirm a change "
              "on a real image or on the ground.",
        notebook=None),
    "small": dict(
        title="Small objects: cars, boats, single trees, roof material",
        verdict="Not a fit. These are smaller than the SR pixel (2.5 m).",
        best=("hr",), best_text="real high-resolution images",
        lines=["SR cannot recover what Sentinel-2 did not record. At this size, what stage 2 "
               "shows is invented.",
               "SR can still tell you where to buy real images."],
        watch="A convincing picture is not evidence.",
        notebook=None),
    "other": dict(
        title="Something else",
        verdict="Not measured. Use the rules of thumb below and plan a test.",
        best=(), best_text="unknown: test it",
        lines=["Find the measured task that is closest to yours and read its line too."],
        watch="",
        notebook=None),
}
RULES = [
    "<b>Size.</b> 10 m and more: try free Sentinel-2 first. 2.5 to 10 m: SR may help, test it. "
    "Under 2.5 m: real images.",
    "<b>Sharper is not more accurate.</b> Strong stage 2 looks best and never measured best "
    "by a clear margin.",
    "<b>Outlines or texture?</b> Tasks that need outlines (fields, single buildings) did best "
    "with stage 1. Tasks that need texture (settlements, land cover) did best with stage 2.",
    "<b>A screening layer.</b> Use SR to decide where to look; confirm on a real image or on "
    "the ground before anyone acts.",
    "<b>The model matters as much as the image.</b> The same image gave different results "
    "with different models.",
]


def _agreement(choice, row):
    key = _CHOICE_KEY.get(choice)
    if not key or not row["best"]:
        return ""
    if key in row["best"]:
        return (f"<b>You chose {choice}: the measurements agree.</b> How would you show that "
                "it also holds in your country?")
    order = ["s2", "stage1", "t100", "t250", "t500", "hr"]
    best = [order.index(b) for b in row["best"]]
    if order.index(key) > max(best):
        how = ("you chose a stronger image than the one that measured best. Does the extra "
               "detail hold up, or does it only look better?")
    elif order.index(key) < min(best):
        how = ("you chose a more careful image than the one that measured best. What did "
               "you distrust?")
    else:
        how = "you chose a different image from the one that measured best."
    return (f"<b>You chose {choice}; measured best: {row['best_text']}.</b> So {how} Your "
            "eyes and the measurements disagree: which test would settle it?")


BASELINES = ["plain Sentinel-2 (does SR add anything?)",
             "real high-resolution images for a sample (how far from the best?)",
             "the way we decide today (is it better than now?)"]
VERDICTS = ["pilot it", "test it first", "not a fit"]


def cheat_panel():
    """Part 5: opens once the team has chosen an image in part 4. The cheat sheet for the
    kind of task, how it compares with the team's choice, and the team's test plan."""
    holder = W.VBox([W.HTML(_note("&#128274; Opens once you have chosen an image in part 4 "
                                  "(“Which image would you use?”).", "#8a6d00"))])
    state = {}

    def build():
        task = W.Dropdown(options=[(r["title"], k) for k, r in CHEAT.items()], value="other",
                          description="Our task is closest to", layout=W.Layout(width="760px"),
                          style={"description_width": "170px"})
        starter = next((s for s in STARTERS if s["title"] == ANSWERS.get("starter")), None)
        if starter:
            task.value = starter["task"]
        sheet = W.HTML()

        def draw(_=None):
            r = CHEAT[task.value]
            ANSWERS["task_type"] = r["title"]
            S["cheat"] = r
            parts = [f"<div style='font-size:15px'><b>{r['verdict']}</b></div>",
                     f"<p style='margin:6px 0'><b>Measured best:</b> {r['best_text']}</p>",
                     "<ul style='margin:4px 0'>" + "".join(f"<li>{x}</li>" for x in r["lines"])
                     + "</ul>"]
            if r["watch"]:
                parts.append(f"<p style='margin:6px 0'><b>Watch out:</b> {r['watch']}</p>")
            parts.append(f"<p style='margin:6px 0'><b>Your size:</b> "
                         f"{_size_rule(float(ANSWERS.get('size_m', 5)))}</p>")
            agree = _agreement(ANSWERS.get("LEVEL"), r)
            if agree:
                parts.append(f"<p style='margin:8px 0; padding:6px 10px; background:#fff7e0'>"
                             f"{agree}</p>")
            if r["notebook"]:
                t, link = MEASURED[r["notebook"]]
                parts.append("<p style='margin:6px 0'><b>Try your minimum value against our "
                             f"data:</b> the notebook <a href='{link}' target='_blank'>{t}</a> "
                             "lets you set your own budget and your own rule for a hit. Its "
                             "result is measured, on that place.</p>")
            rules = "<ul style='margin:4px 0'>" + "".join(f"<li>{x}</li>" for x in RULES) + "</ul>"
            sheet.value = _card("".join(parts)) + _card("<b>Rules of thumb</b>" + rules, "#777")
        task.observe(draw, names="value")
        plan = ask(("TEST_DATA", "What would you check the map against?", "text",
                    "200 blocks our field teams validated in 2025"),
                   ("TEST_BASELINE", "What would you compare it with? (tick all that apply)",
                    "checks", BASELINES),
                   ("TEST_PASS", "The pass mark (from your minimum value in part 1)", "text",
                    "at least 10 hits in 20 visits, in two different cities"),
                   ("NEEDS", "What do you need to start? (data, people, partners, money)",
                    "text", "labels from the housing office; two weeks of an analyst"),
                   show=False)
        state["draw"] = draw
        ui = W.VBox([task, sheet, W.HTML("<br><b>Next steps: how would you test it at "
                                         "home?</b>"), plan])
        ui.task, ui.sheet, ui.fields = task, sheet, plan.fields
        UI["cheat"] = ui
        draw()
        return ui

    def refresh():
        if not ANSWERS.get("LEVEL"):
            return
        if "draw" not in state:
            try:
                holder.children = (build(),)
            except Exception as exc:
                holder.children = (W.HTML(_error(exc)),)
        else:
            state["draw"]()
    ANSWER_REFRESH["cheat"] = refresh
    UI["cheat_holder"] = holder
    display(holder)
    refresh()


# ---- part 6: the slide card --------------------------------------------------------------------

def scenario_card_text():
    lines = ["=" * 78]
    for k, v in canvas_dict().items():
        lines.append(f"{k.replace('_', ' ').upper():<16}{v}")
    lines.append(f"{'SMALLEST THING':<16}{ANSWERS.get('size_m', 0):g} m")
    lines += ["-" * 78, f"{'AREA':<16}{ANSWERS.get('area', '(none chosen)')}"]
    info = S.get("scene_info") or {}
    if info.get("scenes"):
        days = sorted({s["date"] for s in info["scenes"]})
        lines.append(f"{'SENTINEL-2':<16}{len(info['scenes'])} scenes on {len(days)} days, "
                     f"{days[0]} to {days[-1]}")
    if info.get("warnings"):
        lines.append(f"{'WARNINGS':<16}" + ", ".join(
            w if isinstance(w, str) else w.get("code", "?") for w in info["warnings"]))
    if S.get("money"):
        lines.append(f"{'MONEY':<16}{S['money'][0]}")
        lines += [f"  - {x}" for x in S["money"][1:]]
    lines += ["-" * 78,
              f"{'on Sentinel-2 we see it:':<28}{_said('SEE_S2')}",
              f"{'we would use:':<28}{_said('LEVEL')}",
              f"{'because it shows:':<28}{_said('WHY')}",
              f"{'invented or wrong:':<28}{_said('INVENTED')}"]
    r = S.get("cheat")
    if r:
        lines += [f"{'closest measured task:':<28}{r['title']}",
                  f"{'cheat sheet:':<28}{r['verdict']} Measured best: {r['best_text']}."]
        key = _CHOICE_KEY.get(ANSWERS.get("LEVEL"))
        if key and r["best"]:
            lines.append(f"{'our eyes and the sheet:':<28}"
                         + ("agree" if key in r["best"] else "disagree"))
    lines += ["-" * 78, "TEST PLAN",
              f"{'check against:':<28}{_said('TEST_DATA')}",
              f"{'compare with:':<28}" + ("; ".join(ANSWERS.get("TEST_BASELINE") or [])
                                         or "(none ticked)"),
              f"{'pass mark:':<28}{_said('TEST_PASS')}",
              f"{'we need:':<28}{_said('NEEDS')}",
              "-" * 78, f"{'recommendation:':<28}{_said('VERDICT')}"]
    guards = ANSWERS.get("SAFEGUARDS") or []
    lines.append(f"{'safeguards:':<28}" + ("(none ticked)" if not guards else ""))
    lines += [f"  - {g}" for g in guards]
    lines.append("=" * 78)
    return "\n".join(lines)


def scenario_card():
    """Part 6: the recommendation, then a card with everything the slide needs. It follows
    every answer above."""
    form = ask(("VERDICT", "Is super-resolution a fit for your scenario?", VERDICTS),
               ("SAFEGUARDS", "Safeguards you insist on (tick all that apply)", "checks",
                SAFEGUARDS), show=False)
    card = W.HTML()

    def update(_=None):
        card.value = _pre(scenario_card_text())
    ANSWER_REFRESH["scenario_card"] = update
    update()
    ui = W.VBox([form, W.HTML("<b>Your card</b> (copy it onto your team slide; the picture is "
                              "the download of part 4)"), card])
    ui.card, ui.fields = card, form.fields
    UI["scenario_card"] = ui
    display(ui)

In [ ]:
import os

DATA_URL = os.environ.get("HK_DATA_URL", "https://huggingface.co/datasets/adbsr-internal/ssta-hackathon/resolve/main")

get_areas(DATA_URL)

## 1. Your scenario

Agree on **one decision** that a real office makes again and again, and that an image could
inform. Good scenarios are narrow: "which 20 blocks do we visit this quarter" beats "monitor
urban growth".

* **Who decides, what, and how often?** And how do they decide today?
* **What must the image show?** Name the thing and its size. A house is about 8 m, a farm road
  4 m, a car 2 m. Sentinel-2 has 10 m pixels; super-resolution delivers 2.5 m.
* **Success.** Which number would you report to your director? What is the **minimum** you
  need, and why that value? What happens below it: more wasted visits, people left out, a
  wrong notice? Would it then be cheaper to buy real images?

Write your answers before you look at any picture. If your team has no idea after ten minutes,
open "Stuck?" at the bottom, or ask a facilitator for a scenario card.

In [ ]:
canvas()

## 2. Your area and dates

Choose a place where your decision would be made.

* **Ready-made areas.** We ran these before the workshop; their pictures came with the
  notebook. Take the one most like your scenario. You need no key.
* **A place of your own.** Possible, up to 2 km × 2 km, with the API key on your team card.
  The service needs 5 to 10 minutes for the three levels, so **send it before lunch**.

For any area you see **which Sentinel-2 images exist** around your date. Super-resolution
blends several clear images of the same place. In a cloudy season there may be too few.

**Three levels.** Stage 1 is always made. Stage 2 can redraw a little or a lot:

| Image | What it is |
|---|---|
| **Sentinel-2** | The free 10 m image, simply enlarged. The "do nothing" option. |
| **SR stage 1** | Careful: blends the Sentinel-2 images into one sharper image. Invents nothing. |
| **SR stage 2, light** | AI adds a little texture. |
| **SR stage 2, standard** | The setting of the service. |
| **SR stage 2, strong** | AI redraws a lot. Sharpest, and the most invented detail. |

In [ ]:
area_panel()

## 3. What would the images cost?

Your area and how often you need a fresh map come from part 1. Real images and super-resolution
differ in price by a factor of about a thousand, so the question is not "which is cheaper" but
**what must real images save to be worth their price?**

* Set what **one wasted action** costs you: a field visit to the wrong place, a wrong notice.
* Read how many wasted actions a year real images would have to avoid.
* Try the **mixed strategy**: SR everywhere to choose where to look, real images for a small
  share of the area.
* Go back to your **minimum value** in part 1. If SR stays below it, is the extra money for
  real images justified?

In [ ]:
money_panel()

## 4. Look: which image would you use?

Press **Load the pictures**. Zoom to 250 m, move around, and compare the levels on the thing
your decision needs.

* Can you see it on plain Sentinel-2 already?
* What does stage 1 add? What does stage 2 add at each level?
* Find a place you know. Does stage 2 show something that is **not there**? Compare with
  stage 1, which invents nothing: detail that appears only in stage 2 is a guess.
* "Two, large" puts two images side by side.

Then **commit**: which image would you use, and why? Decide before you open part 5.

In [ ]:
look_panel()

## 5. Cheat sheet and next steps

The cheat sheet says what we **measured** for tasks like yours: with experts' labels, in
other places. Choose the task closest to yours.

* Does it agree with what you chose in part 4? If your eyes and the measurements disagree,
  that is the most useful finding of the day.
* The cheat sheet was measured elsewhere, with other models. **How would you test it at
  home?** Fill in the test plan: what you would check the map against, what you would compare
  it with, and the pass mark that follows from your minimum value.
* If one of our measured tasks fits your scenario, open its notebook and try your budget and
  your rule there.

In [ ]:
cheat_panel()

## 6. Your slide

Choose your recommendation and tick your safeguards. The card collects everything and updates
by itself.

**Your slide** has four parts:
1. **The scenario**: who decides what, what the image must show, your minimum value.
2. **What you found**: the area, the cost, the image you would use (add the picture from part 4).
3. **The cheat sheet**: does it agree with you?
4. **Recommendation and test plan**: *pilot it*, *test it first* or *not a fit*; how you would
   test it and what you need.

In [ ]:
scenario_card()

---
Contains modified Copernicus Sentinel data. Super-resolution: ADB SSTA model (stage 1 Sen4x,
stage 2 diffusion refiner). The starting scenarios are invented for this workshop. Workshop material:
please do not redistribute the data packages.